<a href="https://colab.research.google.com/github/ubermanproductionsva-jpg/helix-rna-folding-engine/blob/main/HELIX_v2_FULL_INTEGRATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HELIX v2 Full Integration Notebook

This notebook is the **full integration build** for HELIX reconstruction work.

It includes:

- filename-only discovery
- competition-style CSV loading
- metadata integration
- FASTA manifest workflow
- geometry + topology engine
- reconstruction benchmarks
- quality-weighted model training
- submission scaffolding

Use in **Google Colab** or **Kaggle**.

## Required files

### Core
- `BASELINE_DELTA_TABLE_CLEAN_STRICT.parquet`
- `BASELINE_DELTA_TABLE_CLEAN_STRICT__AR1.parquet`
- `df_model_aligned.parquet` or `CLEAN_SUBSET_LABELS_STRICT.parquet`

### Metadata
- `rna_metadata.csv`

### Optional but recommended
- `BASELINE_FEATURE_TABLE_CLEAN_STRICT.parquet`
- `BASELINE_HGB_VALID_PREDICTIONS_STRICT.csv`
- `.fasta` / `.fa` / `.MSA.fasta`

### Competition
- `train_sequences.csv`
- `validation_sequences.csv`
- `train_labels.csv`
- `validation_labels.csv`
- `test_sequences.csv`
- `sample_submission.csv`

In [8]:
# 1 — MOUNT DRIVE + VERIFY KAGGLE_PROJECT_HUB (SHORTCUT METHOD)

from google.colab import drive
drive.mount('/content/drive')

import os

BASE = "/content/drive/MyDrive"
TARGET = "KAGGLE_PROJECT_HUB"

ROOT = os.path.join(BASE, TARGET)

print("🔍 Checking:", ROOT)

if os.path.exists(ROOT):
    print("✅ FOUND ROOT:", ROOT)

    print("\n📁 CONTENTS:")
    for f in os.listdir(ROOT):
        print(" -", f)
else:
    print("❌ NOT FOUND — Did you add shortcut to MyDrive?")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔍 Checking: /content/drive/MyDrive/KAGGLE_PROJECT_HUB
✅ FOUND ROOT: /content/drive/MyDrive/KAGGLE_PROJECT_HUB

📁 CONTENTS:
 - tab_schemas.csv
 - A_SCHEMA_VALIDATION__v1.json
 - LICENSE.txt
 - A_DIRECTORY_DISCOVERY__v1.json
 - models.md
 - 1AL5_c1 (1).json
 - Kaggle contest_260109_005139 (1).txt
 - FOUNDATION__PIPELINE_FREEZE__v1.json
 - notebooks
 - ROOT_FILENAMES_SNAPSHOT_20260104_135414.csv
 - wl002_vs_wl001_comparison (3).csv
 - 1AFX_c1.json
 - submission (4).csv
 - rna_metadata.csv
 - 1A4T_c1.json
 - RUN_POINTER__B_20260211T152517Z.json
 - submission_v6_curve_prior.csv
 - A_BOOTSTRAP_RECORD.json
 - datasets.md
 - model_instances.md
 - configuration.md
 - B_TEMPLATE_CANDIDATES__B_20260211T061918Z.jsonl
 - kaggle contest part 1 (1).pdf
 - FOUNDATION__PIPELINE_FREEZE__v1.1.json
 - raw
 - B_MSA_MANIFEST__B_20260211T185543Z.json
 - wl002_vs_wl001_comparison (2

In [9]:
# CELL 2 — FULL FILE AUDIT

import os
from collections import defaultdict

file_summary = defaultdict(int)
file_paths = []

for root, dirs, files in os.walk(ROOT):
    for file in files:
        ext = file.lower().split('.')[-1]
        file_summary[ext] += 1
        file_paths.append(os.path.join(root, file))

print("📊 FILE TYPE SUMMARY:")
for k, v in sorted(file_summary.items(), key=lambda x: -x[1]):
    print(f"{k}: {v}")

print("\nTotal files:", len(file_paths))

📊 FILE TYPE SUMMARY:
json: 7011
fasta: 6893
csv: 852
cif: 585
txt: 214
md: 22
a1_witness_pending: 16
py: 9
docx: 6
jsonl: 3
pdf: 3
xlsx: 3
ipynb: 2
tmp: 2
tsv: 1
license: 1
rst: 1
gitignore (1): 1
gitignore: 1
npy: 1
npz: 1
parquet: 1

Total files: 15629


In [10]:
# 3 — STRUCTURED FOLDER EXPLORER (DEEP SCAN + SMART SUMMARY)

import os
from collections import defaultdict

ROOT = "/content/drive/MyDrive/KAGGLE_PROJECT_HUB"

KEY_EXTENSIONS = ["parquet", "csv", "json", "fasta", "msa", "cif"]

summary = defaultdict(int)
important_files = []

print("🔍 SCANNING ROOT:", ROOT)
print("--------------------------------------------------")

for root, dirs, files in os.walk(ROOT):

    level = root.replace(ROOT, "").count(os.sep)
    indent = " " * (level * 2)

    print(f"{indent}📁 {os.path.basename(root)}")

    subindent = " " * ((level + 1) * 2)

    for f in files:
        ext = f.lower().split('.')[-1]
        summary[ext] += 1

        # flag important files
        if ext in KEY_EXTENSIONS:
            important_files.append(os.path.join(root, f))

        # only print key files to reduce noise
        if ext in ["parquet", "csv", "fasta", "msa"]:
            print(f"{subindent}📄 {f}")

print("\n--------------------------------------------------")
print("📊 FILE TYPE SUMMARY:")

for k, v in sorted(summary.items(), key=lambda x: -x[1]):
    print(f"{k}: {v}")

print("\n🧬 IMPORTANT FILE SAMPLE:")
for f in important_files[:25]:
    print("👉", f)

Streaming output truncated to the last 5000 lines.
      📄 Nk44Rg.fasta
      📄 Nk8xTw.fasta
      📄 NFVHMA.fasta
      📄 OVFJVw.fasta
      📄 MU44Ug.fasta
      📄 OUc5Qw.fasta
      📄 Mk9JVQ.fasta
      📄 MVk5NQ.fasta
      📄 MkI2Rw.fasta
      📄 OFI4TQ.fasta
      📄 NlpWSQ.fasta
      📄 Mk5VRw.fasta
      📄 NkFXRA.fasta
      📄 NU5TNA.fasta
      📄 OFo5Sw.fasta
      📄 NFdDRQ.fasta
      📄 NlFOUg.fasta
      📄 N1QzSg.fasta
      📄 NU9EVg.fasta
      📄 N1ROWQ.fasta
      📄 Mks5Ng.fasta
      📄 MlpVRQ.fasta
      📄 OFpEQw.fasta
      📄 MlhMSQ.fasta
      📄 NUZDSg.fasta
      📄 MVJDNw.fasta
      📄 OE9QVA.fasta
      📄 OEcyRA.fasta
      📄 MlBDVg.fasta
      📄 NFdDNg.fasta
      📄 OE9PMA.fasta
      📄 M0YzMA.fasta
      📄 OFFSSw.fasta
      📄 Mk80Mw.fasta
      📄 MkxBQw.fasta
      📄 OFkwRA.fasta
      📄 NFY3Ug.fasta
      📄 NFY1RQ.fasta
      📄 OEYwTg.fasta
      📄 NFUyNw.fasta
      📄 N0JLUA.fasta
      📄 NVUzMA.fasta
      📄 OFpUVQ.fasta
      📄 N1MxSA.fasta
      📄 NURHRg.fasta
    

In [11]:
# 4 — TARGETED DIRECTORY INSPECTION

import os

TARGET_DIRS = [
    "02_extracted",
    "03_normalized",
    "04_features",
    "05_models",
    "07_motifs"
]

ROOT = "/content/drive/MyDrive/KAGGLE_PROJECT_HUB"

for d in TARGET_DIRS:
    path = os.path.join(ROOT, d)

    print("\n==================================================")
    print(f"📂 {d}")
    print("PATH:", path)

    if not os.path.exists(path):
        print("❌ NOT FOUND")
        continue

    files = os.listdir(path)
    print(f"📊 FILE COUNT: {len(files)}")

    # show first 20 files
    for f in files[:20]:
        print(" -", f)


📂 02_extracted
PATH: /content/drive/MyDrive/KAGGLE_PROJECT_HUB/02_extracted
📊 FILE COUNT: 0

📂 03_normalized
PATH: /content/drive/MyDrive/KAGGLE_PROJECT_HUB/03_normalized
📊 FILE COUNT: 1
 - registry

📂 04_features
PATH: /content/drive/MyDrive/KAGGLE_PROJECT_HUB/04_features
📊 FILE COUNT: 1
 - 04_engines

📂 05_models
PATH: /content/drive/MyDrive/KAGGLE_PROJECT_HUB/05_models
📊 FILE COUNT: 0

📂 07_motifs
PATH: /content/drive/MyDrive/KAGGLE_PROJECT_HUB/07_motifs
📊 FILE COUNT: 0


In [12]:
# 5 — SMART FILE PREVIEW

import pandas as pd
import json

def preview_file(path):
    print("\n==================================================")
    print("📄 FILE:", path)

    try:
        if path.endswith(".csv"):
            df = pd.read_csv(path, nrows=5)
            print(df.head())
            print("\nColumns:", list(df.columns))

        elif path.endswith(".parquet"):
            df = pd.read_parquet(path)
            print(df.head())
            print("\nColumns:", list(df.columns))

        elif path.endswith(".json"):
            with open(path, "r") as f:
                data = json.load(f)
            print("Keys:", list(data.keys())[:20])

        else:
            print("Preview not supported")

    except Exception as e:
        print("Error:", e)

# pick some important files automatically
sample_files = [f for f in important_files if any(x in f for x in ["DELTA", "FEATURE", "LABEL", "MSA"])]

for f in sample_files[:10]:
    preview_file(f)


📄 FILE: /content/drive/MyDrive/KAGGLE_PROJECT_HUB/B_MSA_MANIFEST__B_20260211T185543Z.json
Keys: ['artifact_hash', 'artifact_type', 'created_utc', 'env_fingerprint_hash', 'file_count', 'msa_merkle_root', 'msa_root', 'records', 'run_id', 'schema_version', 'total_bytes']

📄 FILE: /content/drive/MyDrive/KAGGLE_PROJECT_HUB/B_MSA_MANIFEST__B_20260211T185543Z__v3.json
Keys: ['artifact_hash', 'artifact_type', 'created_utc', 'file_count', 'manifest_version', 'msa_merkle_root', 'msa_root', 'records', 'run_id', 'schema_version']

📄 FILE: /content/drive/MyDrive/KAGGLE_PROJECT_HUB/B_MSA_MANIFEST__B_20260211T185543Z__v2.json
Keys: ['artifact_hash', 'artifact_type', 'created_utc', 'env_fingerprint_hash', 'file_count', 'msa_merkle_root', 'msa_root', 'origin_tracking', 'records', 'run_id', 'schema_version', 'total_bytes']

📄 FILE: /content/drive/MyDrive/KAGGLE_PROJECT_HUB/B_MSA_MANIFEST__B_20260211T152517Z.json
Keys: ['artifact_hash', 'artifact_type', 'created_utc', 'env_fingerprint_hash', 'file_count

In [13]:
# 6 — BUILD DELTA SOURCE FROM CIF FILES

import os
import numpy as np
import pandas as pd

ROOT = "/content/drive/MyDrive/KAGGLE_PROJECT_HUB"

cif_files = []

for root, dirs, files in os.walk(ROOT):
    for f in files:
        if f.endswith(".cif"):
            cif_files.append(os.path.join(root, f))

print("🧬 CIF files found:", len(cif_files))

# 🔹 SIMPLE CIF PARSER (ATOM COORDS)
def extract_coords_from_cif(path):
    coords = []

    with open(path, "r") as f:
        for line in f:
            if line.startswith("ATOM"):
                parts = line.split()
                try:
                    x = float(parts[10])
                    y = float(parts[11])
                    z = float(parts[12])
                    coords.append([x, y, z])
                except:
                    continue

    return np.array(coords)

rows = []

for i, f in enumerate(cif_files[:50]):  # limit for RAM safety

    coords = extract_coords_from_cif(f)

    if len(coords) < 3:
        continue

    deltas = coords[1:] - coords[:-1]

    for j in range(len(deltas)):
        rows.append({
            "file": os.path.basename(f),
            "idx": j,
            "x": coords[j][0],
            "y": coords[j][1],
            "z": coords[j][2],
            "dx": deltas[j][0],
            "dy": deltas[j][1],
            "dz": deltas[j][2]
        })

SOURCE = pd.DataFrame(rows)

print("✅ SOURCE BUILT:", SOURCE.shape)
SOURCE.head()

🧬 CIF files found: 585
✅ SOURCE BUILT: (1612122, 8)


,file,idx,x,y,z,dx,dy,dz
0,9ash.cif,0,180.807,217.691,273.532,-0.438,0.707,-1.197
1,9ash.cif,1,180.369,218.398,272.335,-0.324,-0.981,-1.121
2,9ash.cif,2,180.045,217.417,271.214,0.674,-1.015,-0.163
3,9ash.cif,3,180.719,216.402,271.051,0.716,2.994,0.831
4,9ash.cif,4,181.435,219.396,271.882,0.111,1.224,0.892


In [14]:
# 7 — GEOMETRY FEATURE ENGINEERING

import numpy as np

df = SOURCE.copy()

# curvature (angle between vectors)
v1 = df[["dx","dy","dz"]].values[:-1]
v2 = df[["dx","dy","dz"]].values[1:]

dot = np.sum(v1*v2, axis=1)
norm = np.linalg.norm(v1, axis=1) * np.linalg.norm(v2, axis=1)

curvature = np.arccos(np.clip(dot / (norm + 1e-8), -1, 1))
curvature = np.append(curvature, curvature[-1])

df["curvature"] = curvature

# torsion (simplified proxy)
df["torsion"] = np.gradient(df["curvature"])

# neighbors
df["delta_dx_prev"] = df["dx"].shift(1)
df["delta_dx_next"] = df["dx"].shift(-1)

df.fillna(0, inplace=True)

SOURCE = df

print("✅ FEATURES READY:", SOURCE.shape)

✅ FEATURES READY: (1612122, 12)


In [15]:
# 8 — BUILD FEATURE MATRIX

import numpy as np

df = SOURCE.copy()

# 🎯 feature columns (NOW VALID)
feature_cols = [
    "x", "y", "z",
    "curvature", "torsion",
    "delta_dx_prev", "delta_dx_next"
]

target_cols = ["dx", "dy", "dz"]

# drop any weird rows
df = df.replace([np.inf, -np.inf], np.nan).dropna()

X = df[feature_cols].values
Y = df[target_cols].values

# weights (can be improved later)
W = np.ones(len(df))

print("✅ X shape:", X.shape)
print("✅ Y shape:", Y.shape)

✅ X shape: (1612122, 7)
✅ Y shape: (1612122, 3)


In [16]:
# 9 — MEMORY SAFE SAMPLING

MAX_ROWS = 120000  # safe for colab

if len(X) > MAX_ROWS:
    print(f"⚠️ Reducing dataset: {len(X)} → {MAX_ROWS}")

    idx = np.random.choice(len(X), MAX_ROWS, replace=False)

    X = X[idx]
    Y = Y[idx]
    W = W[idx]

print("✅ Final training size:", len(X))

⚠️ Reducing dataset: 1612122 → 120000
✅ Final training size: 120000


In [17]:
# 10 — TRAIN / VALID SPLIT

from sklearn.model_selection import train_test_split

X_train, X_valid, Y_train, Y_valid, W_train, W_valid = train_test_split(
    X, Y, W,
    test_size=0.2,
    random_state=42
)

print("✅ Split complete")

✅ Split complete


In [18]:
# 11 — RANDOM FOREST DELTA MODEL

from sklearn.ensemble import RandomForestRegressor

rf_dx = RandomForestRegressor(n_estimators=60, max_depth=12, n_jobs=-1)
rf_dy = RandomForestRegressor(n_estimators=60, max_depth=12, n_jobs=-1)
rf_dz = RandomForestRegressor(n_estimators=60, max_depth=12, n_jobs=-1)

rf_dx.fit(X_train, Y_train[:, 0])
rf_dy.fit(X_train, Y_train[:, 1])
rf_dz.fit(X_train, Y_train[:, 2])

rf_triplet = (rf_dx, rf_dy, rf_dz)

print("✅ RF model trained")

✅ RF model trained


In [19]:
# 12 — HIST GRADIENT BOOSTING MODEL

from sklearn.ensemble import HistGradientBoostingRegressor

hgb_dx = HistGradientBoostingRegressor(max_iter=150)
hgb_dy = HistGradientBoostingRegressor(max_iter=150)
hgb_dz = HistGradientBoostingRegressor(max_iter=150)

hgb_dx.fit(X_train, Y_train[:, 0])
hgb_dy.fit(X_train, Y_train[:, 1])
hgb_dz.fit(X_train, Y_train[:, 2])

hgb_triplet = (hgb_dx, hgb_dy, hgb_dz)

print("✅ HGB model trained")

✅ HGB model trained


In [20]:
# 13 — DELTA ERROR EVALUATION

def eval_model(triplet, name):
    dx, dy, dz = triplet

    pred = np.column_stack([
        dx.predict(X_valid),
        dy.predict(X_valid),
        dz.predict(X_valid)
    ])

    err = np.linalg.norm(pred - Y_valid, axis=1)

    print(f"\n📊 {name}")
    print("Mean:", err.mean())
    print("Median:", np.median(err))
    print("P90:", np.percentile(err, 90))

eval_model(rf_triplet, "RF")
eval_model(hgb_triplet, "HGB")


📊 RF
Mean: 1.9745026234878775
Median: 1.5039257915583082
P90: 3.248481009346243

📊 HGB
Mean: 1.9893500881552488
Median: 1.5161050189155143
P90: 3.248793107359635


In [21]:
# 14 — DRIFT-AWARE RECONSTRUCTION

def reconstruct_sequence(coords_start, deltas):
    coords = [coords_start]

    for d in deltas:
        next_coord = coords[-1] + d
        coords.append(next_coord)

    return np.array(coords)

In [22]:
# 15 — MODEL RECONSTRUCTION TEST

def reconstruct_from_model(df, triplet):
    dx_model, dy_model, dz_model = triplet

    Xg = df[feature_cols].values

    pred_deltas = np.column_stack([
        dx_model.predict(Xg),
        dy_model.predict(Xg),
        dz_model.predict(Xg)
    ])

    coords_true = df[["x","y","z"]].values

    coords_pred = reconstruct_sequence(coords_true[0], pred_deltas)

    min_len = min(len(coords_pred), len(coords_true))

    err = np.linalg.norm(
        coords_pred[:min_len] - coords_true[:min_len],
        axis=1
    )

    return err

In [23]:
# 16 — RECONSTRUCTION EVALUATION

df_sample = SOURCE.sample(5000).sort_values("idx")

rf_err = reconstruct_from_model(df_sample, rf_triplet)
hgb_err = reconstruct_from_model(df_sample, hgb_triplet)

print("\n🧬 RF RECON")
print("Mean:", rf_err.mean())
print("Median:", np.median(rf_err))
print("P90:", np.percentile(rf_err, 90))

print("\n🧬 HGB RECON")
print("Mean:", hgb_err.mean())
print("Median:", np.median(hgb_err))
print("P90:", np.percentile(hgb_err, 90))


🧬 RF RECON
Mean: 214.53324602430993
Median: 205.60830802590166
P90: 355.55965004712465

🧬 HGB RECON
Mean: 211.91707117860847
Median: 201.74574460207077
P90: 346.0507816418733


In [24]:
# 17 — SAVE SOURCE TO DISK (MAKE IT REAL)

save_path = "/content/SOURCE_DEBUG.parquet"

SOURCE.to_parquet(save_path)

print("✅ SOURCE saved to:", save_path)

✅ SOURCE saved to: /content/SOURCE_DEBUG.parquet


In [25]:
# 18 — DATA LINEAGE CHECK

print("📊 SOURCE INFO")
print("Rows:", len(SOURCE))
print("Columns:", list(SOURCE.columns))

print("\n📄 SAMPLE:")
print(SOURCE.head())

print("\n📊 DELTA STATS")
d = np.linalg.norm(SOURCE[["dx","dy","dz"]].values, axis=1)

print("Mean:", d.mean())
print("Median:", np.median(d))
print("P90:", np.percentile(d, 90))

📊 SOURCE INFO
Rows: 1612122
Columns: ['file', 'idx', 'x', 'y', 'z', 'dx', 'dy', 'dz', 'curvature', 'torsion', 'delta_dx_prev', 'delta_dx_next']

📄 SAMPLE:
       file  idx        x        y        z     dx     dy     dz  curvature  \
0  9ash.cif    0  180.807  217.691  273.532 -0.438  0.707 -1.197   1.207222   
1  9ash.cif    1  180.369  218.398  272.335 -0.324 -0.981 -1.121   1.032913   
2  9ash.cif    2  180.045  217.417  271.214  0.674 -1.015 -0.163   2.327797   
3  9ash.cif    3  180.719  216.402  271.051  0.716  2.994  0.831   0.386317   
4  9ash.cif    4  181.435  219.396  271.882  0.111  1.224  0.892   1.170465   

    torsion  delta_dx_prev  delta_dx_next  
0 -0.174309          0.000         -0.324  
1  0.560288         -0.438          0.674  
2 -0.323298         -0.324          0.716  
3 -0.578666          0.674          0.111  
4  0.498328          0.716          0.970  

📊 DELTA STATS
Mean: 2.1987850736016723
Median: 1.5227780534273432
P90: 3.362965317891021


In [26]:
# 19 — ANCHORED RECONSTRUCTION

def anchored_reconstruction(df, triplet, anchor_every=25):
    dx_model, dy_model, dz_model = triplet

    Xg = df[feature_cols].values
    true_coords = df[["x","y","z"]].values

    pred_deltas = np.column_stack([
        dx_model.predict(Xg),
        dy_model.predict(Xg),
        dz_model.predict(Xg)
    ])

    coords = [true_coords[0]]

    for i in range(len(pred_deltas)):
        next_coord = coords[-1] + pred_deltas[i]

        # 🔥 ANCHOR RESET
        if i % anchor_every == 0:
            next_coord = true_coords[i]

        coords.append(next_coord)

    coords = np.array(coords)

    min_len = min(len(coords), len(true_coords))

    err = np.linalg.norm(coords[:min_len] - true_coords[:min_len], axis=1)

    return err

In [27]:
# 20 — RUN ANCHORED RECON

df_sample = SOURCE.sample(5000).sort_values("idx")

rf_err = anchored_reconstruction(df_sample, rf_triplet)
hgb_err = anchored_reconstruction(df_sample, hgb_triplet)

print("\n🧬 RF ANCHORED")
print("Mean:", rf_err.mean())
print("Median:", np.median(rf_err))
print("P90:", np.percentile(rf_err, 90))

print("\n🧬 HGB ANCHORED")
print("Mean:", hgb_err.mean())
print("Median:", np.median(hgb_err))
print("P90:", np.percentile(hgb_err, 90))


🧬 RF ANCHORED
Mean: 234.38819438321545
Median: 213.25706215400277
P90: 454.50056972776014

🧬 HGB ANCHORED
Mean: 234.4487869146794
Median: 212.72653557899724
P90: 454.66229217381203


In [28]:
# 21 — STEP DISTRIBUTION BY FILE

df_check = SOURCE.copy()

df_check["step"] = np.linalg.norm(
    df_check[["dx","dy","dz"]].values, axis=1
)

print(df_check.groupby("file")["step"].mean().describe())

count    33.000000
mean      2.244369
std       0.156346
min       1.946203
25%       2.132482
50%       2.239634
75%       2.387819
max       2.541143
Name: step, dtype: float64


In [29]:
# 22 — FILTER REALISTIC STEPS

df_fixed = SOURCE.copy()

df_fixed["step"] = np.linalg.norm(
    df_fixed[["dx","dy","dz"]].values, axis=1
)

df_fixed = df_fixed[
    (df_fixed["step"] > 4.5) &
    (df_fixed["step"] < 7.5)
]

print("✅ FILTERED SHAPE:", df_fixed.shape)
print("Mean step:", df_fixed["step"].mean())

✅ FILTERED SHAPE: (97085, 13)
Mean step: 5.76614783939976


In [30]:
# 23 — RETRAIN ON FIXED DATA

feature_cols = [
    "idx",
    "curvature",
    "torsion",
    "delta_dx_prev",
    "delta_dx_next"
]

df = df_fixed.dropna().copy()

X = df[feature_cols].values
Y = df[["dx","dy","dz"]].values

from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

In [31]:
# 24 — C1′-ONLY CIF PARSER (CRITICAL FIX)

def extract_c1_coords_from_cif(file_path):
    coords = []

    with open(file_path, "r") as f:
        for line in f:
            if line.startswith("ATOM"):
                parts = line.split()

                atom_name = parts[3]

                # 🔥 ONLY C1′
                if atom_name in ["C1'", "C1*"]:
                    x = float(parts[10])
                    y = float(parts[11])
                    z = float(parts[12])

                    coords.append([x, y, z])

    return np.array(coords)

In [35]:
# 25 — ROBUST CIF PARSER (ATOM TABLE EXTRACTION)

import os
import numpy as np
import pandas as pd

def load_cif_atom_table(file_path):
    """
    Parse mmCIF atom-site records into a DataFrame using loop_ headers.
    Designed for _atom_site style tables.
    """
    headers = []
    rows = []
    in_atom_loop = False
    collecting_headers = False

    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    i = 0
    while i < len(lines):
        line = lines[i].strip()

        # detect loop start
        if line == "loop_":
            j = i + 1
            loop_headers = []

            while j < len(lines):
                s = lines[j].strip()
                if s.startswith("_atom_site."):
                    loop_headers.append(s)
                    j += 1
                else:
                    break

            if loop_headers:
                headers = loop_headers
                in_atom_loop = True
                i = j

                # collect rows until next loop/header block break
                while i < len(lines):
                    s = lines[i].strip()

                    if (
                        s == ""
                        or s == "loop_"
                        or s.startswith("#")
                        or (s.startswith("_") and not s.startswith("_atom_site."))
                    ):
                        break

                    tokens = lines[i].strip().split()
                    if len(tokens) >= len(headers):
                        rows.append(tokens[:len(headers)])
                    i += 1

                break

        i += 1

    if not headers or not rows:
        return pd.DataFrame()

    df_atoms = pd.DataFrame(rows, columns=headers)
    return df_atoms

# test on first CIF
sample_cif = cif_files[0]
df_atoms = load_cif_atom_table(sample_cif)

print("✅ Atom table shape:", df_atoms.shape)
print("Columns:")
print(df_atoms.columns.tolist())

if len(df_atoms) > 0:
    display(df_atoms.head(10))
else:
    print("⚠️ Atom table empty")

✅ Atom table shape: (22309, 21)
Columns:
['_atom_site.group_PDB', '_atom_site.id', '_atom_site.type_symbol', '_atom_site.label_atom_id', '_atom_site.label_alt_id', '_atom_site.label_comp_id', '_atom_site.label_asym_id', '_atom_site.label_entity_id', '_atom_site.label_seq_id', '_atom_site.pdbx_PDB_ins_code', '_atom_site.Cartn_x', '_atom_site.Cartn_y', '_atom_site.Cartn_z', '_atom_site.occupancy', '_atom_site.B_iso_or_equiv', '_atom_site.pdbx_formal_charge', '_atom_site.auth_seq_id', '_atom_site.auth_comp_id', '_atom_site.auth_asym_id', '_atom_site.auth_atom_id', '_atom_site.pdbx_PDB_model_num']


,_atom_site.group_PDB,_atom_site.id,_atom_site.type_symbol,_atom_site.label_atom_id,_atom_site.label_alt_id,_atom_site.label_comp_id,_atom_site.label_asym_id,_atom_site.label_entity_id,_atom_site.label_seq_id,_atom_site.pdbx_PDB_ins_code,...,_atom_site.Cartn_y,_atom_site.Cartn_z,_atom_site.occupancy,_atom_site.B_iso_or_equiv,_atom_site.pdbx_formal_charge,_atom_site.auth_seq_id,_atom_site.auth_comp_id,_atom_site.auth_asym_id,_atom_site.auth_atom_id,_atom_site.pdbx_PDB_model_num
0,ATOM,1,N,N,.,MET,A,1,1,?,...,217.691,273.532,1.00,44.23,?,1,MET,A,N,1
1,ATOM,2,C,CA,.,MET,A,1,1,?,...,218.398,272.335,1.00,45.94,?,1,MET,A,CA,1
2,ATOM,3,C,C,.,MET,A,1,1,?,...,217.417,271.214,1.00,43.61,?,1,MET,A,C,1
3,ATOM,4,O,O,.,MET,A,1,1,?,...,216.402,271.051,1.00,41.89,?,1,MET,A,O,1
4,ATOM,5,C,CB,.,MET,A,1,1,?,...,219.396,271.882,1.00,43.57,?,1,MET,A,CB,1
5,ATOM,6,C,CG,.,MET,A,1,1,?,...,220.620,272.774,1.00,43.07,?,1,MET,A,CG,1
6,ATOM,7,S,SD,.,MET,A,1,1,?,...,221.945,272.031,1.00,52.50,?,1,MET,A,SD,1
7,ATOM,8,C,CE,.,MET,A,1,1,?,...,223.139,273.365,1.00,44.20,?,1,MET,A,CE,1
8,ATOM,9,N,N,.,ASP,A,1,2,?,...,217.730,270.444,1.00,38.28,?,2,ASP,A,N,1
9,ATOM,10,C,CA,.,ASP,A,1,2,?,...,216.808,269.427,1.00,38.98,?,2,ASP,A,CA,1


In [36]:
# 26 — DISCOVER ATOM / COORD COLUMNS

assert len(df_atoms) > 0, "❌ Atom table is empty; parser did not work"

candidate_atom_cols = [c for c in df_atoms.columns if "label_atom_id" in c or "auth_atom_id" in c or c.endswith(".type_symbol")]
candidate_x_cols = [c for c in df_atoms.columns if c.endswith(".Cartn_x")]
candidate_y_cols = [c for c in df_atoms.columns if c.endswith(".Cartn_y")]
candidate_z_cols = [c for c in df_atoms.columns if c.endswith(".Cartn_z")]
candidate_seq_cols = [c for c in df_atoms.columns if "label_seq_id" in c or "auth_seq_id" in c]
candidate_comp_cols = [c for c in df_atoms.columns if "label_comp_id" in c or "auth_comp_id" in c]
candidate_chain_cols = [c for c in df_atoms.columns if "label_asym_id" in c or "auth_asym_id" in c]

print("Atom columns:", candidate_atom_cols)
print("X columns:", candidate_x_cols)
print("Y columns:", candidate_y_cols)
print("Z columns:", candidate_z_cols)
print("Seq columns:", candidate_seq_cols)
print("Residue columns:", candidate_comp_cols)
print("Chain columns:", candidate_chain_cols)

for col in candidate_atom_cols[:3]:
    print(f"\nUnique values from {col}:")
    print(df_atoms[col].dropna().astype(str).unique()[:30])

Atom columns: ['_atom_site.type_symbol', '_atom_site.label_atom_id', '_atom_site.auth_atom_id']
X columns: ['_atom_site.Cartn_x']
Y columns: ['_atom_site.Cartn_y']
Z columns: ['_atom_site.Cartn_z']
Seq columns: ['_atom_site.label_seq_id', '_atom_site.auth_seq_id']
Residue columns: ['_atom_site.label_comp_id', '_atom_site.auth_comp_id']
Chain columns: ['_atom_site.label_asym_id', '_atom_site.auth_asym_id']

Unique values from _atom_site.type_symbol:
['N' 'C' 'O' 'S' 'P' 'MG']

Unique values from _atom_site.label_atom_id:
['N' 'CA' 'C' 'O' 'CB' 'CG' 'SD' 'CE' 'OD1' 'OD2' 'CD' 'NZ' 'CG1' 'CG2'
 'CD1' 'ND2' 'CD2' 'SG' 'CE1' 'CE2' 'CZ' 'OH' 'NE' 'NH1' 'NH2' 'OG1' 'OG'
 'OE1' 'OE2' 'ND1']

Unique values from _atom_site.auth_atom_id:
['N' 'CA' 'C' 'O' 'CB' 'CG' 'SD' 'CE' 'OD1' 'OD2' 'CD' 'NZ' 'CG1' 'CG2'
 'CD1' 'ND2' 'CD2' 'SG' 'CE1' 'CE2' 'CZ' 'OH' 'NE' 'NH1' 'NH2' 'OG1' 'OG'
 'OE1' 'OE2' 'ND1']


In [37]:
# 27 — BUILD CLEAN SOURCE FROM C1' ONLY

import numpy as np
import pandas as pd
import os

# choose columns discovered from Cell 26
ATOM_COL = "_atom_site.label_atom_id" if "_atom_site.label_atom_id" in df_atoms.columns else (
    "_atom_site.auth_atom_id" if "_atom_site.auth_atom_id" in df_atoms.columns else None
)
X_COL = "_atom_site.Cartn_x" if "_atom_site.Cartn_x" in df_atoms.columns else None
Y_COL = "_atom_site.Cartn_y" if "_atom_site.Cartn_y" in df_atoms.columns else None
Z_COL = "_atom_site.Cartn_z" if "_atom_site.Cartn_z" in df_atoms.columns else None
SEQ_COL = "_atom_site.label_seq_id" if "_atom_site.label_seq_id" in df_atoms.columns else (
    "_atom_site.auth_seq_id" if "_atom_site.auth_seq_id" in df_atoms.columns else None
)
CHAIN_COL = "_atom_site.label_asym_id" if "_atom_site.label_asym_id" in df_atoms.columns else (
    "_atom_site.auth_asym_id" if "_atom_site.auth_asym_id" in df_atoms.columns else None
)

assert ATOM_COL is not None, "❌ Could not identify atom-name column"
assert X_COL is not None and Y_COL is not None and Z_COL is not None, "❌ Could not identify coordinate columns"

all_rows = []

MAX_CIFS = 100  # keep safe for Colab
processed = 0
kept_files = 0

for file_path in cif_files[:MAX_CIFS]:
    dfc = load_cif_atom_table(file_path)
    if len(dfc) == 0:
        continue

    # keep only C1' / C1*
    atom_series = dfc[ATOM_COL].astype(str).str.strip()
    dfc = dfc[atom_series.isin(["C1'", "C1*"])].copy()

    if len(dfc) < 5:
        continue

    # numeric coords
    for c in [X_COL, Y_COL, Z_COL]:
        dfc[c] = pd.to_numeric(dfc[c], errors="coerce")

    dfc = dfc.dropna(subset=[X_COL, Y_COL, Z_COL]).copy()

    # residue order
    if SEQ_COL is not None:
        dfc[SEQ_COL] = pd.to_numeric(dfc[SEQ_COL], errors="coerce")
        dfc = dfc.dropna(subset=[SEQ_COL]).sort_values([CHAIN_COL, SEQ_COL] if CHAIN_COL else [SEQ_COL]).copy()
    else:
        dfc = dfc.reset_index(drop=True).copy()

    # build per-chain deltas
    if CHAIN_COL is None:
        dfc["_tmp_chain"] = "A"
        group_cols = ["_tmp_chain"]
    else:
        group_cols = [CHAIN_COL]

    file_rows_before = len(all_rows)

    for _, g in dfc.groupby(group_cols):
        g = g.reset_index(drop=True).copy()
        coords = g[[X_COL, Y_COL, Z_COL]].values.astype(float)

        if len(coords) < 5:
            continue

        deltas = coords[1:] - coords[:-1]

        for i in range(len(deltas)):
            all_rows.append({
                "file": os.path.basename(file_path),
                "chain": str(g.iloc[i][group_cols[0]]),
                "idx": int(i),
                "x": float(coords[i][0]),
                "y": float(coords[i][1]),
                "z": float(coords[i][2]),
                "dx": float(deltas[i][0]),
                "dy": float(deltas[i][1]),
                "dz": float(deltas[i][2]),
            })

    processed += 1
    if len(all_rows) > file_rows_before:
        kept_files += 1

SOURCE_CLEAN = pd.DataFrame(all_rows)

print("✅ Processed CIFs:", processed)
print("✅ Kept CIFs with C1' rows:", kept_files)
print("✅ CLEAN SOURCE shape:", SOURCE_CLEAN.shape)

if len(SOURCE_CLEAN) > 0:
    display(SOURCE_CLEAN.head())
else:
    print("⚠️ Still empty — then the atom naming scheme differs and we use Cell 26 output to adjust.")

✅ Processed CIFs: 0
✅ Kept CIFs with C1' rows: 0
✅ CLEAN SOURCE shape: (0, 0)
⚠️ Still empty — then the atom naming scheme differs and we use Cell 26 output to adjust.


In [38]:
# 29 — CLASSIFY CIFS AS RNA-LIKE VS PROTEIN-LIKE

import os
import pandas as pd

RNA_RESNAMES = {
    "A", "C", "G", "U", "I",
    "ADE", "CYT", "GUA", "URI", "URA"
}

PROTEIN_MARKERS = {
    "MET", "ASP", "GLU", "LYS", "ARG", "LEU", "ILE", "VAL", "SER", "THR",
    "ASN", "GLN", "TYR", "TRP", "PHE", "HIS", "ALA", "GLY", "PRO", "CYS"
}

def classify_cif_content(file_path):
    df_atoms = load_cif_atom_table(file_path)
    if len(df_atoms) == 0:
        return {
            "file": os.path.basename(file_path),
            "rows": 0,
            "rna_hits": 0,
            "protein_hits": 0,
            "label_comp_examples": [],
            "class": "EMPTY"
        }

    comp_col = "_atom_site.label_comp_id" if "_atom_site.label_comp_id" in df_atoms.columns else (
        "_atom_site.auth_comp_id" if "_atom_site.auth_comp_id" in df_atoms.columns else None
    )

    if comp_col is None:
        return {
            "file": os.path.basename(file_path),
            "rows": len(df_atoms),
            "rna_hits": 0,
            "protein_hits": 0,
            "label_comp_examples": [],
            "class": "NO_COMP_COLUMN"
        }

    residues = set(df_atoms[comp_col].astype(str).str.strip().unique())
    rna_hits = len(residues & RNA_RESNAMES)
    protein_hits = len(residues & PROTEIN_MARKERS)

    if rna_hits > 0 and protein_hits == 0:
        cls = "RNA"
    elif protein_hits > 0 and rna_hits == 0:
        cls = "PROTEIN"
    elif rna_hits > 0 and protein_hits > 0:
        cls = "MIXED"
    else:
        cls = "UNKNOWN"

    return {
        "file": os.path.basename(file_path),
        "rows": len(df_atoms),
        "rna_hits": rna_hits,
        "protein_hits": protein_hits,
        "label_comp_examples": sorted(list(residues))[:15],
        "class": cls
    }

audit_rows = []
for fp in cif_files[:200]:
    try:
        audit_rows.append(classify_cif_content(fp))
    except Exception as e:
        audit_rows.append({
            "file": os.path.basename(fp),
            "rows": -1,
            "rna_hits": 0,
            "protein_hits": 0,
            "label_comp_examples": [f"ERROR: {e}"],
            "class": "ERROR"
        })

df_cif_audit = pd.DataFrame(audit_rows)
display(df_cif_audit["class"].value_counts(dropna=False))
display(df_cif_audit.head(20))

,count
class,
MIXED,93
EMPTY,70
RNA,33
UNKNOWN,3
PROTEIN,1


,file,rows,rna_hits,protein_hits,label_comp_examples,class
0,9ash.cif,22309,4,20,"[A, ALA, ARG, ASN, ASP, ATP, C, CYS, G, GLN, G...",MIXED
1,9nl2.cif,12138,4,20,"[A, ALA, ARG, ASN, ASP, C, CYS, DA, DC, DG, DT...",MIXED
2,5u0q.cif,706,4,0,"[A, C, CO, DT, G, HOH, MG, U]",RNA
3,1am0.cif,4480,4,0,"[A, AMP, C, G, U]",RNA
4,5n61.cif,48547,3,20,"[A, ALA, ARG, ASN, ASP, C, CYS, DA, DC, DG, DT...",MIXED
5,9g26.cif,33973,4,20,"[3DR, A, ALA, ARG, ASN, ASP, C, CYS, DA, DC, D...",MIXED
6,5zem.cif,939,4,0,"[1MA, A, C, G, HOH, U]",RNA
7,1arj.cif,19140,4,1,"[A, ARG, C, G, U]",MIXED
8,169d.cif,635,2,0,"[C, DA, DC, DG, DT, G]",RNA
9,3lob.cif,7299,3,20,"[A, ALA, ARG, ASN, ASP, C, CYS, GLN, GLU, GLY,...",MIXED


In [39]:
# 30 — LIST RNA-LIKE CIFS

rna_like_files = []

for fp in cif_files:
    try:
        result = classify_cif_content(fp)
        if result["class"] in ["RNA", "MIXED"]:
            rna_like_files.append(fp)
    except Exception:
        pass

print("✅ RNA-like CIF count:", len(rna_like_files))
for x in rna_like_files[:50]:
    print(x)

✅ RNA-like CIF count: 389
/content/drive/MyDrive/KAGGLE_PROJECT_HUB/01_data/structures/9ash.cif
/content/drive/MyDrive/KAGGLE_PROJECT_HUB/01_data/structures/9nl2.cif
/content/drive/MyDrive/KAGGLE_PROJECT_HUB/01_data/structures/5u0q.cif
/content/drive/MyDrive/KAGGLE_PROJECT_HUB/01_data/structures/1am0.cif
/content/drive/MyDrive/KAGGLE_PROJECT_HUB/01_data/structures/5n61.cif
/content/drive/MyDrive/KAGGLE_PROJECT_HUB/01_data/structures/9g26.cif
/content/drive/MyDrive/KAGGLE_PROJECT_HUB/01_data/structures/5zem.cif
/content/drive/MyDrive/KAGGLE_PROJECT_HUB/01_data/structures/1arj.cif
/content/drive/MyDrive/KAGGLE_PROJECT_HUB/01_data/structures/169d.cif
/content/drive/MyDrive/KAGGLE_PROJECT_HUB/01_data/structures/3lob.cif
/content/drive/MyDrive/KAGGLE_PROJECT_HUB/01_data/structures/8rqk.cif
/content/drive/MyDrive/KAGGLE_PROJECT_HUB/01_data/structures/9hdr.cif
/content/drive/MyDrive/KAGGLE_PROJECT_HUB/01_data/structures/6nu2.cif
/content/drive/MyDrive/KAGGLE_PROJECT_HUB/01_data/structures/7xl

In [40]:
# 31 — AUDIT _c1.JSON FILES

import json
import os
from collections import Counter

ROOT = "/content/drive/MyDrive/KAGGLE_PROJECT_HUB"

c1_json_files = []
for root, dirs, files in os.walk(ROOT):
    for f in files:
        if f.lower().endswith("_c1.json") or "_c1" in f.lower():
            c1_json_files.append(os.path.join(root, f))

print("✅ _c1 JSON files found:", len(c1_json_files))
for p in c1_json_files[:25]:
    print(p)

sample_keys = []
for p in c1_json_files[:10]:
    try:
        with open(p, "r", encoding="utf-8", errors="ignore") as f:
            obj = json.load(f)
        if isinstance(obj, dict):
            sample_keys.append({
                "file": os.path.basename(p),
                "top_keys": list(obj.keys())[:20]
            })
        else:
            sample_keys.append({
                "file": os.path.basename(p),
                "top_keys": [type(obj).__name__]
            })
    except Exception as e:
        sample_keys.append({
            "file": os.path.basename(p),
            "top_keys": [f"ERROR: {e}"]
        })

display(pd.DataFrame(sample_keys))

✅ _c1 JSON files found: 60
/content/drive/MyDrive/KAGGLE_PROJECT_HUB/1AL5_c1 (1).json
/content/drive/MyDrive/KAGGLE_PROJECT_HUB/1AFX_c1.json
/content/drive/MyDrive/KAGGLE_PROJECT_HUB/1A4T_c1.json
/content/drive/MyDrive/KAGGLE_PROJECT_HUB/1AC3_c1.json
/content/drive/MyDrive/KAGGLE_PROJECT_HUB/104D_c1.json
/content/drive/MyDrive/KAGGLE_PROJECT_HUB/1A1T_c1.json
/content/drive/MyDrive/KAGGLE_PROJECT_HUB/161D_c1.json
/content/drive/MyDrive/KAGGLE_PROJECT_HUB/1ANR_c1.json
/content/drive/MyDrive/KAGGLE_PROJECT_HUB/1AJF_c1.json
/content/drive/MyDrive/KAGGLE_PROJECT_HUB/165D_c1.json
/content/drive/MyDrive/KAGGLE_PROJECT_HUB/1A9L_c1.json
/content/drive/MyDrive/KAGGLE_PROJECT_HUB/176D_c1.json
/content/drive/MyDrive/KAGGLE_PROJECT_HUB/169D_c1.json
/content/drive/MyDrive/KAGGLE_PROJECT_HUB/1AM0_c1.json
/content/drive/MyDrive/KAGGLE_PROJECT_HUB/1AL5_c1.json
/content/drive/MyDrive/KAGGLE_PROJECT_HUB/100D_c1.json
/content/drive/MyDrive/KAGGLE_PROJECT_HUB/124D_c1.json
/content/drive/MyDrive/KAGGLE_PROJ

,file,top_keys
0,1AL5_c1 (1).json,"[A, B]"
1,1AFX_c1.json,[A]
2,1A4T_c1.json,[A]
3,1AC3_c1.json,[B]
4,104D_c1.json,"[A, B]"
5,1A1T_c1.json,[B]
6,161D_c1.json,"[A, B]"
7,1ANR_c1.json,[A]
8,1AJF_c1.json,[A]
9,165D_c1.json,"[A, B]"


In [40]:
# 32 — INSPECT ONE _c1.JSON DEEPLY

import json
import os
from pprint import pprint

assert len(c1_json_files) > 0, "❌ No _c1 JSON files found"

sample_json = c1_json_files[0]
print("Sample _c1 file:", sample_json)

with open(sample_json, "r", encoding="utf-8", errors="ignore") as f:
    obj = json.load(f)

print("Top-level type:", type(obj))

if isinstance(obj, dict):
    print("Top-level keys:")
    print(list(obj.keys())[:50])

    for k, v in obj.items():
        print(f"\nKEY: {k}")
        print("TYPE:", type(v))
        if isinstance(v, list):
            print("LEN:", len(v))
            print("SAMPLE:", v[:2])
        elif isinstance(v, dict):
            print("DICT KEYS:", list(v.keys())[:20])
        else:
            print("VALUE:", v)
        break

elif isinstance(obj, list):
    print("List length:", len(obj))
    print("First item:")
    pprint(obj[0] if len(obj) > 0 else None)
else:
    print(obj)

In [42]:
import json

path = "/content/drive/MyDrive/KAGGLE_PROJECT_HUB/1AM0_c1.json"

with open(path, "r") as f:
    data = json.load(f)

print(type(data))
print(data.keys())

first_chain = list(data.keys())[0]
print("First chain:", first_chain)

chain_data = data[first_chain]
print(type(chain_data))
print(len(chain_data))

print("Sample entries:")
for i in range(min(5, len(chain_data))):
    print(chain_data[i])

<class 'dict'>
dict_keys(['A'])
First chain: A
<class 'list'>
16
Sample entries:
{'resid': 6, 'resname': 'G', 'x': -14.842, 'y': -6.443, 'z': 6.416, 'altloc': '\x00'}
{'resid': 7, 'resname': 'G', 'x': -12.408, 'y': -9.153, 'z': 8.798, 'altloc': '\x00'}
{'resid': 8, 'resname': 'G', 'x': -8.085, 'y': -8.795, 'z': 10.899, 'altloc': '\x00'}
{'resid': 9, 'resname': 'A', 'x': -6.215, 'y': -4.109, 'z': 14.151, 'altloc': '\x00'}
{'resid': 10, 'resname': 'A', 'x': -3.626, 'y': -3.502, 'z': 10.517, 'altloc': '\x00'}


In [44]:
# 1. Extract numeric coordinates
coords = np.array([
    [r["x"], r["y"], r["z"]]
    for r in chain_data
], dtype=np.float32)

# 2. Compute deltas
deltas = coords[1:] - coords[:-1]

# 3. Step lengths
steps = np.linalg.norm(deltas, axis=1)

# 4. Stats
print("Residue count:", len(coords))
print("Mean step:", steps.mean())
print("Median step:", np.median(steps))
print("P90:", np.percentile(steps, 90))

# 5. Sanity preview
print("\nFirst 5 steps:", steps[:5])

Residue count: 16
Mean step: 5.996894
Median step: 5.6393056
P90: 7.8798084

First 5 steps: [4.352285  4.819823  6.0025826 4.503035  6.4742393]


In [45]:
resids = [r["resid"] for r in chain_data]

print("First 10 resids:", resids[:10])
print("Is strictly increasing:", all(
    resids[i] < resids[i+1] for i in range(len(resids)-1)
))

First 10 resids: [6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
Is strictly increasing: True


In [46]:
# 37 — BUILD SOURCE_CLEAN FROM _c1.JSON FILES + GLOBAL GEOMETRY VALIDATION

import os
import re
import json
import math
import glob
import numpy as np
import pandas as pd

ROOT = "/content/drive/MyDrive/KAGGLE_PROJECT_HUB"
OUT_DIR = os.path.join(ROOT, "04_features")
os.makedirs(OUT_DIR, exist_ok=True)

json_paths = sorted(glob.glob(os.path.join(ROOT, "*_c1*.json")))
print("🔍 _c1 JSON files found:", len(json_paths))

rows = []
file_summaries = []

def normalize_target_id_from_filename(path: str) -> str:
    base = os.path.basename(path)
    stem = os.path.splitext(base)[0]
    stem = re.sub(r"_c1.*$", "", stem, flags=re.IGNORECASE)
    stem = stem.replace(" ", "")
    stem = stem.upper()
    return stem

def safe_float(x):
    try:
        return float(x)
    except Exception:
        return np.nan

for path in json_paths:
    target_id = normalize_target_id_from_filename(path)

    try:
        with open(path, "r") as f:
            data = json.load(f)
    except Exception as e:
        print(f"⚠️ Failed to load {path}: {e}")
        continue

    if not isinstance(data, dict):
        print(f"⚠️ Skipping non-dict JSON: {path}")
        continue

    for chain, chain_entries in data.items():
        if not isinstance(chain_entries, list) or len(chain_entries) == 0:
            continue

        chain_rows = []

        for rec in chain_entries:
            if not isinstance(rec, dict):
                continue

            resid = rec.get("resid", None)
            resname = rec.get("resname", None)
            x = safe_float(rec.get("x", np.nan))
            y = safe_float(rec.get("y", np.nan))
            z = safe_float(rec.get("z", np.nan))
            altloc = rec.get("altloc", "\x00")

            if pd.isna(x) or pd.isna(y) or pd.isna(z):
                continue

            try:
                resid_num = int(resid)
            except Exception:
                continue

            chain_rows.append({
                "source_file": os.path.basename(path),
                "source_path": path,
                "target_id": target_id,
                "chain": str(chain),
                "residue_index": resid_num,
                "resname": str(resname) if resname is not None else None,
                "x": float(x),
                "y": float(y),
                "z": float(z),
                "altloc": altloc,
            })

        if len(chain_rows) == 0:
            continue

        chain_df = pd.DataFrame(chain_rows).sort_values(
            ["residue_index", "x", "y", "z"]
        ).reset_index(drop=True)

        # Remove exact duplicate residue_index rows by keeping first occurrence.
        # This is conservative and prevents duplicate-residue jumps.
        dup_count = int(chain_df["residue_index"].duplicated().sum())
        if dup_count > 0:
            chain_df = chain_df.drop_duplicates(
                subset=["residue_index"], keep="first"
            ).reset_index(drop=True)

        # Sequential deltas within chain only
        chain_df["dx"] = chain_df["x"].diff()
        chain_df["dy"] = chain_df["y"].diff()
        chain_df["dz"] = chain_df["z"].diff()
        chain_df["step"] = np.sqrt(
            chain_df["dx"]**2 + chain_df["dy"]**2 + chain_df["dz"]**2
        )

        resids = chain_df["residue_index"].tolist()
        is_strictly_increasing = all(
            resids[i] < resids[i + 1] for i in range(len(resids) - 1)
        ) if len(resids) > 1 else True

        finite_steps = chain_df["step"].dropna().values
        step_mean = float(np.mean(finite_steps)) if len(finite_steps) else np.nan
        step_median = float(np.median(finite_steps)) if len(finite_steps) else np.nan
        step_p90 = float(np.percentile(finite_steps, 90)) if len(finite_steps) else np.nan

        file_summaries.append({
            "source_file": os.path.basename(path),
            "target_id": target_id,
            "chain": str(chain),
            "residue_count": int(len(chain_df)),
            "step_count": int(len(finite_steps)),
            "step_mean": step_mean,
            "step_median": step_median,
            "step_p90": step_p90,
            "strictly_increasing": bool(is_strictly_increasing),
            "duplicate_residue_rows_removed": dup_count,
        })

        rows.extend(chain_df.to_dict(orient="records"))

source_clean = pd.DataFrame(rows)
summary_df = pd.DataFrame(file_summaries)

if source_clean.empty:
    raise RuntimeError("❌ SOURCE_CLEAN build failed: no rows extracted from _c1 JSON files.")

source_clean = source_clean.sort_values(
    ["target_id", "chain", "residue_index"]
).reset_index(drop=True)

# Global validation
global_steps = source_clean["step"].dropna().values
global_mean = float(np.mean(global_steps)) if len(global_steps) else np.nan
global_median = float(np.median(global_steps)) if len(global_steps) else np.nan
global_p90 = float(np.percentile(global_steps, 90)) if len(global_steps) else np.nan

valid_step_window = 5.7 <= global_mean <= 6.2
strict_order_ok = bool(summary_df["strictly_increasing"].all()) if not summary_df.empty else False

print("\n🧬 SOURCE_CLEAN BUILD COMPLETE")
print("Rows:", len(source_clean))
print("Targets:", source_clean["target_id"].nunique())
print("Chains:", source_clean[["target_id", "chain"]].drop_duplicates().shape[0])

print("\n📊 GLOBAL STEP STATS")
print("Mean step:", round(global_mean, 6))
print("Median step:", round(global_median, 6))
print("P90:", round(global_p90, 6))

print("\n✅ VALIDATION FLAGS")
print("Step mean in expected RNA window [5.7, 6.2]:", valid_step_window)
print("All chains strictly increasing:", strict_order_ok)

print("\n🧾 SAMPLE")
display(source_clean.head(10))

print("\n🧾 CHAIN SUMMARY SAMPLE")
display(summary_df.head(10))

source_clean_out = os.path.join(OUT_DIR, "SOURCE_CLEAN__C1_JSON.csv")
summary_out = os.path.join(OUT_DIR, "SOURCE_CLEAN__C1_JSON__SUMMARY.csv")

source_clean.to_csv(source_clean_out, index=False)
summary_df.to_csv(summary_out, index=False)

print("\n💾 SAVED")
print(source_clean_out)
print(summary_out)

if not valid_step_window:
    raise RuntimeError(
        f"❌ Global mean step {global_mean:.6f} is outside expected RNA window [5.7, 6.2]. STOP."
    )

if not strict_order_ok:
    raise RuntimeError("❌ One or more chains are not strictly increasing by residue_index. STOP.")

🔍 _c1 JSON files found: 29

🧬 SOURCE_CLEAN BUILD COMPLETE
Rows: 598
Targets: 25
Chains: 34

📊 GLOBAL STEP STATS
Mean step: 5.703313
Median step: 5.445593
P90: 6.663463

✅ VALIDATION FLAGS
Step mean in expected RNA window [5.7, 6.2]: True
All chains strictly increasing: True

🧾 SAMPLE


,source_file,source_path,target_id,chain,residue_index,resname,x,y,z,altloc,dx,dy,dz,step
0,100D_c1.json,/content/drive/MyDrive/KAGGLE_PROJECT_HUB/100D...,100D,A,1,C,-4.489,7.917,6.825, ,NaN,NaN,NaN,NaN
1,100D_c1.json,/content/drive/MyDrive/KAGGLE_PROJECT_HUB/100D...,100D,A,10,G,10.721,-5.691,-2.168, ,15.210,-13.608,-8.993,22.302372
2,100D_c1.json,/content/drive/MyDrive/KAGGLE_PROJECT_HUB/100D...,100D,B,11,C,13.918,3.977,0.766, ,NaN,NaN,NaN,NaN
3,100D_c1.json,/content/drive/MyDrive/KAGGLE_PROJECT_HUB/100D...,100D,B,20,G,-3.782,8.924,17.491, ,-17.700,4.947,16.725,24.849315
4,104D_c1.json,/content/drive/MyDrive/KAGGLE_PROJECT_HUB/104D...,104D,A,1,C,2.239,-4.169,0.198, ,NaN,NaN,NaN,NaN
5,104D_c1.json,/content/drive/MyDrive/KAGGLE_PROJECT_HUB/104D...,104D,A,2,G,6.692,-1.183,1.333, ,4.453,2.986,1.135,5.480295
6,104D_c1.json,/content/drive/MyDrive/KAGGLE_PROJECT_HUB/104D...,104D,A,3,C,7.805,2.878,4.337, ,1.113,4.061,3.004,5.172476
7,104D_c1.json,/content/drive/MyDrive/KAGGLE_PROJECT_HUB/104D...,104D,A,4,G,6.044,5.898,8.459, ,-1.761,3.020,4.122,5.404850
8,104D_c1.json,/content/drive/MyDrive/KAGGLE_PROJECT_HUB/104D...,104D,B,13,C,7.215,2.164,28.931, ,NaN,NaN,NaN,NaN
9,104D_c1.json,/content/drive/MyDrive/KAGGLE_PROJECT_HUB/104D...,104D,B,14,G,10.362,-1.916,26.907, ,3.147,-4.080,-2.024,5.535936



🧾 CHAIN SUMMARY SAMPLE


,source_file,target_id,chain,residue_count,step_count,step_mean,step_median,step_p90,strictly_increasing,duplicate_residue_rows_removed
0,100D_c1.json,100D,A,2,1,22.302372,22.302372,22.302372,True,0
1,100D_c1.json,100D,B,2,1,24.849315,24.849315,24.849315,True,0
2,104D_c1.json,104D,A,4,3,5.352540,5.404850,5.465206,True,0
3,104D_c1.json,104D,B,4,3,5.374772,5.411241,5.510997,True,0
4,124D_c1.json,124D,B,8,7,5.391142,5.402062,5.570929,True,0
5,157D_c1.json,157D,A,12,11,5.441743,5.501288,5.687955,True,0
6,157D_c1.json,157D,B,12,11,5.504484,5.453912,5.907639,True,0
7,161D_c1.json,161D,A,1,0,NaN,NaN,NaN,True,0
8,161D_c1.json,161D,B,1,0,NaN,NaN,NaN,True,0
9,165D_c1.json,165D,A,8,7,5.607820,5.508321,5.987032,True,0



💾 SAVED
/content/drive/MyDrive/KAGGLE_PROJECT_HUB/04_features/SOURCE_CLEAN__C1_JSON.csv
/content/drive/MyDrive/KAGGLE_PROJECT_HUB/04_features/SOURCE_CLEAN__C1_JSON__SUMMARY.csv


In [47]:
# 38 — ENFORCE SEQUENTIAL GEOMETRY (REMOVE DISCONTINUITIES)

import pandas as pd
import numpy as np
import os

SRC_PATH = "/content/drive/MyDrive/KAGGLE_PROJECT_HUB/04_features/SOURCE_CLEAN__C1_JSON.csv"
OUT_PATH = "/content/drive/MyDrive/KAGGLE_PROJECT_HUB/04_features/SOURCE_CLEAN__C1_JSON__SEQUENTIAL.csv"

df = pd.read_csv(SRC_PATH)

df = df.sort_values(["target_id", "chain", "residue_index"]).reset_index(drop=True)

# Compute residue gaps per chain
df["resid_diff"] = df.groupby(["target_id", "chain"])["residue_index"].diff()

# Keep only true sequential steps
df["is_sequential"] = df["resid_diff"] == 1

# Remove non-sequential rows (but keep first row of each chain)
df_clean = df[(df["is_sequential"]) | (df["resid_diff"].isna())].copy()

# Recompute deltas cleanly
df_clean["dx"] = df_clean.groupby(["target_id", "chain"])["x"].diff()
df_clean["dy"] = df_clean.groupby(["target_id", "chain"])["y"].diff()
df_clean["dz"] = df_clean.groupby(["target_id", "chain"])["z"].diff()

df_clean["step"] = np.sqrt(
    df_clean["dx"]**2 + df_clean["dy"]**2 + df_clean["dz"]**2
)

# Global validation again
steps = df_clean["step"].dropna().values

print("\n🧬 SEQUENTIAL CLEAN COMPLETE")
print("Rows (before):", len(df))
print("Rows (after):", len(df_clean))

print("\n📊 NEW STEP STATS")
print("Mean:", steps.mean())
print("Median:", np.median(steps))
print("P90:", np.percentile(steps, 90))

# Save
df_clean.to_csv(OUT_PATH, index=False)

print("\n💾 SAVED:", OUT_PATH)


🧬 SEQUENTIAL CLEAN COMPLETE
Rows (before): 598
Rows (after): 524

📊 NEW STEP STATS
Mean: 5.59789928250928
Median: 5.43331097366738
P90: 6.468937484378473

💾 SAVED: /content/drive/MyDrive/KAGGLE_PROJECT_HUB/04_features/SOURCE_CLEAN__C1_JSON__SEQUENTIAL.csv


In [48]:
# 39 — PHYSICAL STEP FILTER (FINAL GEOMETRY LOCK)

import pandas as pd
import numpy as np

SRC_PATH = "/content/drive/MyDrive/KAGGLE_PROJECT_HUB/04_features/SOURCE_CLEAN__C1_JSON__SEQUENTIAL.csv"
OUT_PATH = "/content/drive/MyDrive/KAGGLE_PROJECT_HUB/04_features/SOURCE_CLEAN__C1_JSON__FINAL.csv"

df = pd.read_csv(SRC_PATH)

# Remove rows without valid steps
df_valid = df.dropna(subset=["step"]).copy()

# Keep only physically plausible RNA backbone steps
LOW, HIGH = 4.5, 8.0

df_valid["valid_step"] = (df_valid["step"] >= LOW) & (df_valid["step"] <= HIGH)

df_final = df_valid[df_valid["valid_step"]].copy()

# Recompute stats
steps = df_final["step"].values

print("\n🧬 FINAL GEOMETRY LOCK")
print("Rows (before):", len(df))
print("Rows (after sequential):", len(df_valid))
print("Rows (final):", len(df_final))

print("\n📊 FINAL STEP STATS")
print("Mean:", steps.mean())
print("Median:", np.median(steps))
print("P90:", np.percentile(steps, 90))
print("Min:", steps.min())
print("Max:", steps.max())

# Save
df_final.to_csv(OUT_PATH, index=False)

print("\n💾 SAVED:", OUT_PATH)


🧬 FINAL GEOMETRY LOCK
Rows (before): 524
Rows (after sequential): 490
Rows (final): 464

📊 FINAL STEP STATS
Mean: 5.56071299287617
Median: 5.436000200959105
P90: 6.35175195368291
Min: 4.503035198618816
Max: 7.978463636565627

💾 SAVED: /content/drive/MyDrive/KAGGLE_PROJECT_HUB/04_features/SOURCE_CLEAN__C1_JSON__FINAL.csv


In [49]:
# 40 — GEOMETRY FEATURE BUILD (CURVATURE + DIRECTION + NORMALIZED DELTAS)

import pandas as pd
import numpy as np

SRC_PATH = "/content/drive/MyDrive/KAGGLE_PROJECT_HUB/04_features/SOURCE_CLEAN__C1_JSON__FINAL.csv"
OUT_PATH = "/content/drive/MyDrive/KAGGLE_PROJECT_HUB/04_features/FEATURE_TABLE__GEOMETRY_V1.csv"

df = pd.read_csv(SRC_PATH)

df = df.sort_values(["target_id", "chain", "residue_index"]).reset_index(drop=True)

# --- Normalize deltas ---
df["step_safe"] = df["step"].replace(0, np.nan)

df["dx_norm"] = df["dx"] / df["step_safe"]
df["dy_norm"] = df["dy"] / df["step_safe"]
df["dz_norm"] = df["dz"] / df["step_safe"]

# --- Previous direction (for curvature) ---
df["dx_prev"] = df.groupby(["target_id", "chain"])["dx"].shift(1)
df["dy_prev"] = df.groupby(["target_id", "chain"])["dy"].shift(1)
df["dz_prev"] = df.groupby(["target_id", "chain"])["dz"].shift(1)

df["step_prev"] = df.groupby(["target_id", "chain"])["step"].shift(1)

# Normalize previous
df["dx_prev_norm"] = df["dx_prev"] / df["step_prev"]
df["dy_prev_norm"] = df["dy_prev"] / df["step_prev"]
df["dz_prev_norm"] = df["dz_prev"] / df["step_prev"]

# --- Curvature proxy (dot product of direction vectors) ---
df["curvature"] = (
    df["dx_norm"] * df["dx_prev_norm"] +
    df["dy_norm"] * df["dy_prev_norm"] +
    df["dz_norm"] * df["dz_prev_norm"]
)

# --- Turn angle (optional, more interpretable) ---
df["turn_angle"] = np.arccos(np.clip(df["curvature"], -1.0, 1.0))

# --- Clean NaNs (edges)
df = df.dropna(subset=[
    "dx_norm", "dy_norm", "dz_norm",
    "dx_prev_norm", "dy_prev_norm", "dz_prev_norm"
]).reset_index(drop=True)

# --- Final feature set
feature_cols = [
    "dx_norm", "dy_norm", "dz_norm",
    "curvature", "turn_angle",
    "step"
]

print("\n🧬 FEATURE BUILD COMPLETE")
print("Rows:", len(df))
print("Features:", feature_cols)

print("\n📊 FEATURE STATS")
display(df[feature_cols].describe())

# Save
df.to_csv(OUT_PATH, index=False)

print("\n💾 SAVED:", OUT_PATH)


🧬 FEATURE BUILD COMPLETE
Rows: 434
Features: ['dx_norm', 'dy_norm', 'dz_norm', 'curvature', 'turn_angle', 'step']

📊 FEATURE STATS


,dx_norm,dy_norm,dz_norm,curvature,turn_angle,step
count,434.000000,434.000000,434.000000,434.000000,434.000000,434.000000
mean,0.037973,0.018085,-0.007214,0.725757,0.686244,5.555624
std,0.586882,0.605789,0.537655,0.335701,0.420351,0.637881
min,-0.996639,-0.992794,-0.988986,-0.785543,0.041825,4.503035
25%,-0.487215,-0.550365,-0.475017,0.724419,0.447913,5.158713
50%,0.066983,0.045828,-0.031612,0.850859,0.553177,5.416378
75%,0.537655,0.607991,0.459500,0.901353,0.760603,5.758642
max,0.994673,0.999370,0.963092,0.999125,2.474369,7.978464



💾 SAVED: /content/drive/MyDrive/KAGGLE_PROJECT_HUB/04_features/FEATURE_TABLE__GEOMETRY_V1.csv


In [51]:
# 41 — TRAIN MODEL + RECONSTRUCT (FIXED INDEXING + CLEAN FILTERING)

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor

SRC_PATH = "/content/drive/MyDrive/KAGGLE_PROJECT_HUB/04_features/FEATURE_TABLE__GEOMETRY_V1.csv"

df = pd.read_csv(SRC_PATH)

# --- Targets (next-step deltas) ---
df["dx_target"] = df.groupby(["target_id", "chain"])["dx"].shift(-1)
df["dy_target"] = df.groupby(["target_id", "chain"])["dy"].shift(-1)
df["dz_target"] = df.groupby(["target_id", "chain"])["dz"].shift(-1)

# --- Keep ONLY rows where targets exist ---
df_model = df.dropna(subset=[
    "dx_target", "dy_target", "dz_target",
    "dx_norm", "dy_norm", "dz_norm",
    "curvature", "turn_angle", "step"
]).reset_index(drop=True)

print("Usable rows:", len(df_model))

# --- Features ---
X_cols = ["dx_norm", "dy_norm", "dz_norm", "curvature", "turn_angle", "step"]

X = df_model[X_cols].values
y_dx = df_model["dx_target"].values
y_dy = df_model["dy_target"].values
y_dz = df_model["dz_target"].values

# --- Train models ---
model_dx = RandomForestRegressor(n_estimators=100, random_state=42)
model_dy = RandomForestRegressor(n_estimators=100, random_state=42)
model_dz = RandomForestRegressor(n_estimators=100, random_state=42)

model_dx.fit(X, y_dx)
model_dy.fit(X, y_dy)
model_dz.fit(X, y_dz)

# --- Reconstruction ---
recon_errors = []

for (target_id, chain), group in df_model.groupby(["target_id", "chain"]):

    group = group.sort_values("residue_index").reset_index(drop=True)

    if len(group) < 3:
        continue

    coords_true = group[["x", "y", "z"]].values

    X_group = group[X_cols].values

    pred_dx = model_dx.predict(X_group)
    pred_dy = model_dy.predict(X_group)
    pred_dz = model_dz.predict(X_group)

    coords_pred = [coords_true[0]]

    for i in range(1, len(group)):
        dx = pred_dx[i-1]
        dy = pred_dy[i-1]
        dz = pred_dz[i-1]

        next_coord = coords_pred[-1] + np.array([dx, dy, dz])
        coords_pred.append(next_coord)

    coords_pred = np.array(coords_pred)

    error = np.mean(np.linalg.norm(coords_pred - coords_true, axis=1))
    recon_errors.append(error)

# --- Report ---
print("\n🧬 RECONSTRUCTION RESULTS")
print("Chains evaluated:", len(recon_errors))

if len(recon_errors) > 0:
    print("Mean error:", np.mean(recon_errors))
    print("Median error:", np.median(recon_errors))
    print("P90:", np.percentile(recon_errors, 90))
else:
    print("❌ No valid chains for evaluation")

Usable rows: 404

🧬 RECONSTRUCTION RESULTS
Chains evaluated: 27
Mean error: 3.2668239513849824
Median error: 2.785589051329371
P90: 5.685254801721437


In [52]:
# 42 — GLOBAL CORRECTION UTILITIES (DISTANCE MATRIX + MDS + KABSCH ALIGNMENT)

import numpy as np
import pandas as pd

def pairwise_distances(coords: np.ndarray) -> np.ndarray:
    coords = np.asarray(coords, dtype=np.float64)
    diffs = coords[:, None, :] - coords[None, :, :]
    return np.sqrt(np.sum(diffs * diffs, axis=-1))

def classical_mds_from_dist(D: np.ndarray, eps: float = 1e-8):
    """
    Reconstruct 3D coordinates from an NxN Euclidean distance matrix using classical MDS.
    Returns:
        coords_mds: [N, 3]
        evals: eigenvalues of centered Gram matrix (descending)
        neg_eig_ratio: fraction of negative eigenvalue mass
    """
    D = np.asarray(D, dtype=np.float64)
    n = D.shape[0]
    if D.ndim != 2 or D.shape[0] != D.shape[1]:
        raise ValueError("Distance matrix must be square [N, N].")

    D2 = D ** 2
    J = np.eye(n) - np.ones((n, n)) / n
    B = -0.5 * J @ D2 @ J

    evals, evecs = np.linalg.eigh(B)
    order = np.argsort(evals)[::-1]
    evals = evals[order]
    evecs = evecs[:, order]

    pos_evals = np.clip(evals[:3], a_min=0.0, a_max=None)
    coords_mds = evecs[:, :3] @ np.diag(np.sqrt(pos_evals + eps))

    neg_mass = float(np.abs(evals[evals < 0]).sum())
    total_mass = float(np.abs(evals).sum()) + eps
    neg_eig_ratio = neg_mass / total_mass

    return coords_mds, evals, neg_eig_ratio

def kabsch_align(pred: np.ndarray, ref: np.ndarray) -> np.ndarray:
    """
    Align pred -> ref using rigid transform (no scaling).
    Returns aligned pred coordinates.
    """
    pred = np.asarray(pred, dtype=np.float64)
    ref = np.asarray(ref, dtype=np.float64)

    pred_centroid = pred.mean(axis=0, keepdims=True)
    ref_centroid = ref.mean(axis=0, keepdims=True)

    pred_centered = pred - pred_centroid
    ref_centered = ref - ref_centroid

    H = pred_centered.T @ ref_centered
    U, S, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T

    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1
        R = Vt.T @ U.T

    pred_aligned = pred_centered @ R + ref_centroid
    return pred_aligned

def mean_point_error(a: np.ndarray, b: np.ndarray) -> float:
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    return float(np.mean(np.linalg.norm(a - b, axis=1)))

def distance_matrix_error(a: np.ndarray, b: np.ndarray) -> float:
    Da = pairwise_distances(a)
    Db = pairwise_distances(b)
    return float(np.mean(np.abs(Da - Db)))

In [53]:
# 43 — GLOBAL CORRECTION EVALUATION (LOCAL PATH VS MDS-CORRECTED PATH)

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

SRC_PATH = "/content/drive/MyDrive/KAGGLE_PROJECT_HUB/04_features/FEATURE_TABLE__GEOMETRY_V1.csv"
OUT_CSV = "/content/drive/MyDrive/KAGGLE_PROJECT_HUB/04_features/GLOBAL_CORRECTION_RESULTS__V1.csv"

df = pd.read_csv(SRC_PATH)

# --- next-step targets ---
df["dx_target"] = df.groupby(["target_id", "chain"])["dx"].shift(-1)
df["dy_target"] = df.groupby(["target_id", "chain"])["dy"].shift(-1)
df["dz_target"] = df.groupby(["target_id", "chain"])["dz"].shift(-1)

X_cols = ["dx_norm", "dy_norm", "dz_norm", "curvature", "turn_angle", "step"]

df_model = df.dropna(subset=X_cols + ["dx_target", "dy_target", "dz_target"]).copy()
df_model = df_model.sort_values(["target_id", "chain", "residue_index"]).reset_index(drop=True)

print("Usable rows:", len(df_model))
print("Usable chains:", df_model[["target_id", "chain"]].drop_duplicates().shape[0])

# --- train RF models on all usable rows (same validation regime as prior cell) ---
X = df_model[X_cols].values
y_dx = df_model["dx_target"].values
y_dy = df_model["dy_target"].values
y_dz = df_model["dz_target"].values

model_dx = RandomForestRegressor(n_estimators=100, random_state=42)
model_dy = RandomForestRegressor(n_estimators=100, random_state=42)
model_dz = RandomForestRegressor(n_estimators=100, random_state=42)

model_dx.fit(X, y_dx)
model_dy.fit(X, y_dy)
model_dz.fit(X, y_dz)

rows = []

for (target_id, chain), group in df_model.groupby(["target_id", "chain"]):
    group = group.sort_values("residue_index").reset_index(drop=True)

    # Need enough residues for meaningful distance geometry
    if len(group) < 4:
        continue

    coords_true = group[["x", "y", "z"]].values.astype(np.float64)
    X_group = group[X_cols].values

    pred_dx = model_dx.predict(X_group)
    pred_dy = model_dy.predict(X_group)
    pred_dz = model_dz.predict(X_group)

    # --- Local sequential path reconstruction ---
    coords_local = [coords_true[0].copy()]
    for i in range(1, len(group)):
        step_vec = np.array([pred_dx[i - 1], pred_dy[i - 1], pred_dz[i - 1]], dtype=np.float64)
        coords_local.append(coords_local[-1] + step_vec)
    coords_local = np.asarray(coords_local, dtype=np.float64)

    # Align local path to true coords for fair pointwise comparison
    coords_local_aligned = kabsch_align(coords_local, coords_true)

    # --- Global correction: distance matrix from local path -> MDS ---
    D_local = pairwise_distances(coords_local)
    coords_mds, evals, neg_eig_ratio = classical_mds_from_dist(D_local)
    coords_mds_aligned = kabsch_align(coords_mds, coords_local_aligned)
    coords_mds_to_true = kabsch_align(coords_mds, coords_true)

    # --- Metrics ---
    local_point_error = mean_point_error(coords_local_aligned, coords_true)
    global_point_error = mean_point_error(coords_mds_to_true, coords_true)

    local_dm_error = distance_matrix_error(coords_local_aligned, coords_true)
    global_dm_error = distance_matrix_error(coords_mds_to_true, coords_true)

    rows.append({
        "target_id": target_id,
        "chain": chain,
        "residue_count": int(len(group)),
        "local_point_error": local_point_error,
        "global_point_error": global_point_error,
        "improvement_point_error": local_point_error - global_point_error,
        "local_distance_error": local_dm_error,
        "global_distance_error": global_dm_error,
        "improvement_distance_error": local_dm_error - global_dm_error,
        "neg_eig_ratio": float(neg_eig_ratio),
        "top_eig_1": float(evals[0]) if len(evals) > 0 else np.nan,
        "top_eig_2": float(evals[1]) if len(evals) > 1 else np.nan,
        "top_eig_3": float(evals[2]) if len(evals) > 2 else np.nan,
    })

results = pd.DataFrame(rows)

if results.empty:
    raise RuntimeError("❌ No valid chains were evaluated for global correction.")

print("\n🧬 GLOBAL CORRECTION RESULTS")
print("Chains evaluated:", len(results))

print("\n📊 POINT ERROR")
print("Local mean:", results["local_point_error"].mean())
print("Global mean:", results["global_point_error"].mean())
print("Mean improvement:", results["improvement_point_error"].mean())

print("\n📊 DISTANCE MATRIX ERROR")
print("Local mean:", results["local_distance_error"].mean())
print("Global mean:", results["global_distance_error"].mean())
print("Mean improvement:", results["improvement_distance_error"].mean())

print("\n📊 EUCLIDEANITY / MDS HEALTH")
print("Mean negative eigen ratio:", results["neg_eig_ratio"].mean())
print("P90 negative eigen ratio:", results["neg_eig_ratio"].quantile(0.90))

print("\n🧾 SAMPLE")
display(results.head(10))

results.to_csv(OUT_CSV, index=False)
print("\n💾 SAVED:", OUT_CSV)

Usable rows: 404
Usable chains: 29

🧬 GLOBAL CORRECTION RESULTS
Chains evaluated: 26

📊 POINT ERROR
Local mean: 3.3330357950832235
Global mean: 10.915806479907198
Mean improvement: -7.582770684823974

📊 DISTANCE MATRIX ERROR
Local mean: 2.184256360470303
Global mean: 2.1842563601718314
Mean improvement: 2.98471175125697e-10

📊 EUCLIDEANITY / MDS HEALTH
Mean negative eigen ratio: 3.320189270410754e-16
P90 negative eigen ratio: 7.253950471642772e-16

🧾 SAMPLE


,target_id,chain,residue_count,local_point_error,global_point_error,improvement_point_error,local_distance_error,global_distance_error,improvement_distance_error,neg_eig_ratio,top_eig_1,top_eig_2,top_eig_3
0,124D,B,5,1.733194,8.949234,-7.216041,0.849912,0.849912,7.513714e-10,2.281889e-17,173.759749,17.710135,0.409510
1,157D,A,9,3.375635,7.285464,-3.909829,1.100055,1.100055,2.526614e-10,1.275635e-16,633.956184,158.903890,30.993767
2,157D,B,9,3.728136,3.245331,0.482805,1.159390,1.159390,2.545151e-10,9.787132e-17,643.989758,157.988487,26.110080
3,165D,A,5,1.775458,1.019423,0.756035,0.824613,0.824613,7.455754e-10,1.836544e-17,158.615688,25.580011,0.044668
4,165D,B,5,2.605826,9.354272,-6.748446,0.948655,0.948655,7.514297e-10,3.443969e-17,145.386980,28.264098,0.196035
5,17RA,A,18,2.321320,17.478719,-15.157399,1.875677,1.875677,5.530376e-11,3.868362e-16,1612.983768,328.445461,273.464834
6,1A1T,B,17,2.072139,11.972503,-9.900364,1.120518,1.120518,8.873391e-11,4.423792e-16,1416.270502,279.592299,180.765801
7,1A4D,A,18,1.049562,9.960390,-8.910828,0.523820,0.523820,7.616263e-11,2.542024e-16,5041.531929,386.736081,214.730630
8,1A4D,B,17,0.923292,15.664459,-14.741167,0.670913,0.670913,8.517365e-11,3.381763e-16,4339.585415,304.896834,256.598133
9,1A4T,A,12,2.989719,9.778844,-6.789125,1.900770,1.900770,1.938356e-10,3.342570e-16,440.260223,152.637930,62.417476



💾 SAVED: /content/drive/MyDrive/KAGGLE_PROJECT_HUB/04_features/GLOBAL_CORRECTION_RESULTS__V1.csv


In [54]:
# 44 — HYBRID LOCAL + GLOBAL SMOOTHING

import numpy as np
import pandas as pd

SRC_PATH = "/content/drive/MyDrive/KAGGLE_PROJECT_HUB/04_features/FEATURE_TABLE__GEOMETRY_V1.csv"

df = pd.read_csv(SRC_PATH)

df["dx_target"] = df.groupby(["target_id", "chain"])["dx"].shift(-1)
df["dy_target"] = df.groupby(["target_id", "chain"])["dy"].shift(-1)
df["dz_target"] = df.groupby(["target_id", "chain"])["dz"].shift(-1)

X_cols = ["dx_norm", "dy_norm", "dz_norm", "curvature", "turn_angle", "step"]

df_model = df.dropna(subset=X_cols + ["dx_target", "dy_target", "dz_target"]).copy()
df_model = df_model.sort_values(["target_id", "chain", "residue_index"]).reset_index(drop=True)

from sklearn.ensemble import RandomForestRegressor

X = df_model[X_cols].values
y_dx = df_model["dx_target"].values
y_dy = df_model["dy_target"].values
y_dz = df_model["dz_target"].values

model_dx = RandomForestRegressor(n_estimators=100, random_state=42)
model_dy = RandomForestRegressor(n_estimators=100, random_state=42)
model_dz = RandomForestRegressor(n_estimators=100, random_state=42)

model_dx.fit(X, y_dx)
model_dy.fit(X, y_dy)
model_dz.fit(X, y_dz)

def smooth_coords(coords, alpha=0.2):
    """
    Laplacian smoothing (keeps shape, reduces drift)
    """
    coords = coords.copy()
    for i in range(1, len(coords) - 1):
        coords[i] = coords[i] + alpha * (
            (coords[i-1] + coords[i+1]) / 2 - coords[i]
        )
    return coords

recon_local = []
recon_smooth = []

for (target_id, chain), group in df_model.groupby(["target_id", "chain"]):
    group = group.sort_values("residue_index").reset_index(drop=True)

    if len(group) < 4:
        continue

    coords_true = group[["x", "y", "z"]].values
    Xg = group[X_cols].values

    pred_dx = model_dx.predict(Xg)
    pred_dy = model_dy.predict(Xg)
    pred_dz = model_dz.predict(Xg)

    coords_pred = [coords_true[0]]

    for i in range(1, len(group)):
        step_vec = np.array([pred_dx[i-1], pred_dy[i-1], pred_dz[i-1]])
        coords_pred.append(coords_pred[-1] + step_vec)

    coords_pred = np.array(coords_pred)

    coords_smooth = smooth_coords(coords_pred, alpha=0.25)

    # Align both to truth
    def align(a, b):
        ac = a - a.mean(0)
        bc = b - b.mean(0)
        H = ac.T @ bc
        U, S, Vt = np.linalg.svd(H)
        R = Vt.T @ U.T
        if np.linalg.det(R) < 0:
            Vt[-1] *= -1
            R = Vt.T @ U.T
        return ac @ R + b.mean(0)

    coords_pred = align(coords_pred, coords_true)
    coords_smooth = align(coords_smooth, coords_true)

    err_local = np.mean(np.linalg.norm(coords_pred - coords_true, axis=1))
    err_smooth = np.mean(np.linalg.norm(coords_smooth - coords_true, axis=1))

    recon_local.append(err_local)
    recon_smooth.append(err_smooth)

print("\n🧬 HYBRID CORRECTION RESULTS")
print("Chains:", len(recon_local))

print("\nLOCAL")
print("Mean:", np.mean(recon_local))

print("\nSMOOTHED")
print("Mean:", np.mean(recon_smooth))

print("\nIMPROVEMENT")
print("Δ:", np.mean(recon_local) - np.mean(recon_smooth))


🧬 HYBRID CORRECTION RESULTS
Chains: 26

LOCAL
Mean: 3.3330357950832235

SMOOTHED
Mean: 3.4337156188625504

IMPROVEMENT
Δ: -0.10067982377932694


In [ ]:
# 45 — BUILD GEOMETRY FEATURES FROM COMPETITION LABELS (COLAB LOCAL PATHS)

import pandas as pd
import numpy as np

TRAIN_PATH = "/content/train_labels.csv"
VAL_PATH   = "/content/validation_labels.csv"

OUT_PATH = "/content/FEATURE_TABLE__GEOMETRY_LABELS_V1.csv"

# --- LOAD ---
df_train = pd.read_csv(TRAIN_PATH)
df_val   = pd.read_csv(VAL_PATH)

df = pd.concat([df_train, df_val], ignore_index=True)

print("Loaded rows:", len(df))

# --- DERIVE IDENTIFIERS ---
df["target_id"] = df["ID"].str.split("_").str[0]
df["residue_index"] = df["resid"]

# --- RENAME COORDS ---
df["x"] = df["x_1"]
df["y"] = df["y_1"]
df["z"] = df["z_1"]

# --- SORT (CRITICAL) ---
df = df.sort_values(["target_id", "chain", "copy", "residue_index"]).reset_index(drop=True)

# --- SEQUENTIAL FILTER ---
df["resid_diff"] = df.groupby(["target_id", "chain", "copy"])["residue_index"].diff()
df["is_seq"] = (df["resid_diff"] == 1)

df = df[(df["is_seq"]) | (df["resid_diff"].isna())].copy()

# --- DELTAS ---
df["dx"] = df.groupby(["target_id", "chain", "copy"])["x"].diff()
df["dy"] = df.groupby(["target_id", "chain", "copy"])["y"].diff()
df["dz"] = df.groupby(["target_id", "chain", "copy"])["z"].diff()

df["step"] = np.sqrt(df["dx"]**2 + df["dy"]**2 + df["dz"]**2)

# --- PHYSICAL FILTER ---
df = df[(df["step"].isna()) | ((df["step"] >= 4.5) & (df["step"] <= 8.0))].copy()

# --- NORMALIZED DIRECTION ---
df["step_safe"] = df["step"].replace(0, np.nan)

df["dx_norm"] = df["dx"] / df["step_safe"]
df["dy_norm"] = df["dy"] / df["step_safe"]
df["dz_norm"] = df["dz"] / df["step_safe"]

# --- PREVIOUS STEP ---
df["dx_prev"] = df.groupby(["target_id", "chain", "copy"])["dx"].shift(1)
df["dy_prev"] = df.groupby(["target_id", "chain", "copy"])["dy"].shift(1)
df["dz_prev"] = df.groupby(["target_id", "chain", "copy"])["dz"].shift(1)

df["step_prev"] = df.groupby(["target_id", "chain", "copy"])["step"].shift(1)

df["dx_prev_norm"] = df["dx_prev"] / df["step_prev"]
df["dy_prev_norm"] = df["dy_prev"] / df["step_prev"]
df["dz_prev_norm"] = df["dz_prev"] / df["step_prev"]

# --- CURVATURE ---
df["curvature"] = (
    df["dx_norm"] * df["dx_prev_norm"] +
    df["dy_norm"] * df["dy_prev_norm"] +
    df["dz_norm"] * df["dz_prev_norm"]
)

df["turn_angle"] = np.arccos(np.clip(df["curvature"], -1.0, 1.0))

# --- FINAL CLEAN ---
df_final = df.dropna(subset=[
    "dx_norm", "dy_norm", "dz_norm",
    "dx_prev_norm", "dy_prev_norm", "dz_prev_norm",
    "curvature", "turn_angle", "step"
]).reset_index(drop=True)

# --- REPORT ---
print("\n🧬 LABEL GEOMETRY BUILD COMPLETE")
print("Rows:", len(df_final))
print("Targets:", df_final["target_id"].nunique())
print("Chains:", df_final.groupby(["target_id","chain","copy"]).ngroups)

steps = df_final["step"].values

print("\n📊 STEP STATS")
print("Mean:", steps.mean())
print("Median:", np.median(steps))
print("P90:", np.percentile(steps, 90))
print("Min:", steps.min())
print("Max:", steps.max())

# --- SAVE ---
df_final.to_csv(OUT_PATH, index=False)

print("\n💾 SAVED:", OUT_PATH)

In [41]:
xv cxcxv

SyntaxError: invalid syntax (1244926366.py, line 1)

In [ ]:
# 1 — ENVIRONMENT BOOTSTRAP

import os, re, gc, json, math, time, shutil, random, warnings
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

RUNTIME_IS_KAGGLE = os.path.exists("/kaggle")
RUNTIME_IS_COLAB = os.path.exists("/content")

print("✅ Environment initialized")
print("RUNTIME_IS_KAGGLE:", RUNTIME_IS_KAGGLE)
print("RUNTIME_IS_COLAB:", RUNTIME_IS_COLAB)

In [ ]:
# 2 — DRIVE MOUNT + PROJECT ROOTS

if RUNTIME_IS_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as e:
        print("⚠️ Drive mount skipped:", e)

PROJECT_NAME = "HELIX_TRAINING_READY"
PROJECT_ROOT_COLAB = f"/content/drive/MyDrive/{PROJECT_NAME}"
PROJECT_ROOT_LOCAL = f"/content/{PROJECT_NAME}"
PROJECT_ROOT = PROJECT_ROOT_COLAB if os.path.exists("/content/drive/MyDrive") else PROJECT_ROOT_LOCAL

DIRS = {
    "root": PROJECT_ROOT,
    "registry": f"{PROJECT_ROOT}/00_REGISTRY",
    "truth_coord": f"{PROJECT_ROOT}/01_TRUTH_COORD",
    "delta_truth": f"{PROJECT_ROOT}/02_DELTA_TRUTH",
    "predictions": f"{PROJECT_ROOT}/03_PREDICTIONS",
    "recon": f"{PROJECT_ROOT}/04_RECON_BASELINES",
    "regime": f"{PROJECT_ROOT}/05_REGIME_METADATA",
    "notebooks": f"{PROJECT_ROOT}/06_NOTEBOOKS",
    "governance": f"{PROJECT_ROOT}/07_GOVERNANCE",
    "geometry": f"{PROJECT_ROOT}/08_GEOMETRY_ENGINE",
    "benchmarks": f"{PROJECT_ROOT}/09_BENCHMARKS",
    "fasta": f"{PROJECT_ROOT}/10_FASTA_MANIFESTS",
    "metadata": f"{PROJECT_ROOT}/11_METADATA",
    "models": f"{PROJECT_ROOT}/12_MODELS",
    "submission": f"{PROJECT_ROOT}/13_SUBMISSION",
    "exports": f"{PROJECT_ROOT}/14_EXPORTS",
    "debug": f"{PROJECT_ROOT}/15_DEBUG",
}

for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

print("✅ Project directories ready")
for k, v in DIRS.items():
    print(f"{k}: {v}")

In [ ]:
# 3 — SAFE FILE REGISTRY (LOW-RAM VERSION)

from collections import defaultdict
import os, json

SEARCH_ROOTS = [
    "/content",
    "/content/drive/MyDrive/HELIX_TRAINING_READY"
]

MAX_FILES = 2000  # hard cap to prevent RAM crash

FILE_REGISTRY = defaultdict(list)
DIR_REGISTRY = defaultdict(list)

count = 0

for root in SEARCH_ROOTS:
    if not os.path.exists(root):
        continue

    for dirpath, dirnames, filenames in os.walk(root):
        DIR_REGISTRY[os.path.basename(dirpath)].append(dirpath)

        for f in filenames:
            FILE_REGISTRY[f].append(os.path.join(dirpath, f))
            count += 1

            if count >= MAX_FILES:
                break

        if count >= MAX_FILES:
            break

    if count >= MAX_FILES:
        break

snapshot_path = f"{DIRS['registry']}/FILE_REGISTRY_SNAPSHOT_SAFE.json"
with open(snapshot_path, "w", encoding="utf-8") as f:
    json.dump(dict(FILE_REGISTRY), f, indent=2)

print("✅ SAFE FILE REGISTRY BUILT")
print("Files indexed:", count)
print("Snapshot:", snapshot_path)

In [ ]:
# 4 — CANONICAL RESOLVER

def canonical_sort_key(path: str):
    p = path.lower()
    return (
        0 if "helix_training_ready" in p else 1,
        0 if "/3/20/26 all/" in p else 1,
        0 if "helix_training_ready" in p and "/03_predictions/" in p else 1,
        0 if "helix_dataset_authority" in p else 1,
        0 if "helix_kaggle_dump" in p else 1,
        1 if ".encrypted/" in p else 0,
        len(path),
    )

def resolve_file(filename: str, required_substrings=None, return_all=False):
    matches = FILE_REGISTRY.get(filename, [])
    if required_substrings:
        req = [s.lower() for s in required_substrings]
        matches = [m for m in matches if all(s in m.lower() for s in req)]
    matches = sorted(matches, key=canonical_sort_key)
    if return_all:
        return matches
    return matches[0] if matches else None

def resolve_first_existing(candidates):
    for name in candidates:
        p = resolve_file(name)
        if p:
            return p
    return None

print("✅ Resolver ready")

In [ ]:
# 5 — COMPETITION FILE RESOLUTION

COMP_FILES = {
    "train_sequences": ["train_sequences.csv", "train_sequences (2).csv"],
    "validation_sequences": ["validation_sequences.csv", "validation_sequences (2).csv"],
    "train_labels": ["train_labels.csv", "train_labels (6).csv"],
    "validation_labels": ["validation_labels.csv", "validation_labels (5).csv"],
    "test_sequences": ["test_sequences.csv", "test_sequences (1).csv"],
    "sample_submission": ["sample_submission.csv", "sample_submission (3).csv"],
}

COMP_PATHS = {k: resolve_first_existing(v) for k, v in COMP_FILES.items()}

print("✅ Competition file resolution")
for k, v in COMP_PATHS.items():
    print(f"{k}: {v}")

In [ ]:
# 6 — LOAD COMPETITION FILES

COMP_DATA = {}
for k, p in COMP_PATHS.items():
    if p and os.path.exists(p):
        try:
            COMP_DATA[k] = pd.read_csv(p)
            print(f"✅ Loaded {k}: {COMP_DATA[k].shape}")
        except Exception as e:
            print(f"❌ Failed loading {k}: {e}")
    else:
        print(f"⚠️ Missing {k}")

In [ ]:
# 7 — RESOLVE HELIX CORE DATASETS

HELIX_TARGETS = {
    "df_model_aligned": ["df_model_aligned.parquet"],
    "truth_labels_strict": ["CLEAN_SUBSET_LABELS_STRICT.parquet"],
    "feature_table_strict": ["BASELINE_FEATURE_TABLE_CLEAN_STRICT.parquet"],
    "delta_table_strict": ["BASELINE_DELTA_TABLE_CLEAN_STRICT.parquet"],
    "delta_table_ar1": ["BASELINE_DELTA_TABLE_CLEAN_STRICT__AR1.parquet"],
    "coord_predictions": ["BASELINE_HGB_VALID_PREDICTIONS_STRICT.csv"],
    "metadata_csv": ["rna_metadata.csv"],
    "fasta_parser": ["parse_fasta_py.py", "parse_fasta_py.py"],
    "notebook_current": ["Copy_of_file_breakdown.ipynb", "file_breakdown.ipynb"],
}

HELIX_PATHS = {k: resolve_first_existing(v) for k, v in HELIX_TARGETS.items()}

print("✅ HELIX path resolution")
for k, v in HELIX_PATHS.items():
    print(f"{k}: {v}")

In [ ]:
# 8 — SOURCE RESOLUTION + DELTA FALLBACK (AUTHORITATIVE)

import os
import pandas as pd
import numpy as np

print("🔍 Resolving core datasets...")

# 🔹 SAFE LOADERS
def load_parquet_safe(path, n=200000):
    if path is None or not os.path.exists(path):
        print(f"⚠️ Missing parquet: {path}")
        return None
    df = pd.read_parquet(path)
    if len(df) > n:
        df = df.head(n)
    print(f"✅ Loaded parquet: {path} | Shape: {df.shape}")
    return df

def load_csv_safe(path, n=200000):
    if path is None or not os.path.exists(path):
        print(f"⚠️ Missing csv: {path}")
        return None
    df = pd.read_csv(path)
    if len(df) > n:
        df = df.head(n)
    print(f"✅ Loaded csv: {path} | Shape: {df.shape}")
    return df

# 🔹 LOAD CORE FILES (FROM FILE PANE)
df_model_aligned = load_parquet_safe("/content/df_model_aligned.parquet")
df_meta = load_csv_safe("/content/rna_metadata.csv")

# 🔹 TRUTH
TRUTH = df_model_aligned
assert TRUTH is not None, "❌ No truth dataset available"

# 🔥 DELTA BUILDER (CRITICAL)
def build_delta(df):
    print("⚠️ Building delta from truth (fallback mode)")

    df = df.sort_values(["target_id", "chain", "copy", "residue_index"]).copy()

    df["prev_x"] = df.groupby(["target_id"])["x_1"].shift(1)
    df["prev_y"] = df.groupby(["target_id"])["y_1"].shift(1)
    df["prev_z"] = df.groupby(["target_id"])["z_1"].shift(1)

    df["delta_dx"] = df["x_1"] - df["prev_x"]
    df["delta_dy"] = df["y_1"] - df["prev_y"]
    df["delta_dz"] = df["z_1"] - df["prev_z"]

    df[["delta_dx","delta_dy","delta_dz"]] = df[
        ["delta_dx","delta_dy","delta_dz"]
    ].fillna(0.0)

    return df

# 🔹 BUILD SOURCE (ALWAYS GUARANTEED NOW)
SOURCE = build_delta(TRUTH)

print("\n✅ SOURCE READY:", SOURCE.shape)

In [ ]:
# 9 — DATASET REGISTRY

registry_rows = [
    ["df_model_aligned.parquet", "TRUTH_COORD", "Aligned residue-level truth coordinates", "Evaluation truth", "Absolute coordinates", "Evaluation / diagnostics"],
    ["CLEAN_SUBSET_LABELS_STRICT.parquet", "TRUTH_COORD", "Clean truth-only geometry table", "Truth reference", "Absolute coordinates", "Truth alignment / audits"],
    ["BASELINE_FEATURE_TABLE_CLEAN_STRICT.parquet", "FEATURE_TABLE", "Engineered feature table", "Model feature space", "Feature manifold", "Training / feature analysis"],
    ["BASELINE_DELTA_TABLE_CLEAN_STRICT.parquet", "DELTA_TRUTH", "Primary local motion truth table", "Core reconstruction dataset", "Local delta geometry", "Reconstruction"],
    ["BASELINE_DELTA_TABLE_CLEAN_STRICT__AR1.parquet", "DELTA_TRUTH_AR1", "AR1-enriched delta table", "Advanced reconstruction dataset", "Local delta + temporal geometry", "Curvature / dynamics"],
    ["BASELINE_HGB_VALID_PREDICTIONS_STRICT.csv", "PREDICTED_COORD_OUTPUT", "Direct coordinate prediction artifact", "Failure/control comparison", "Predicted absolute coordinates", "Evaluation"],
    ["rna_metadata.csv", "STRUCTURE_METADATA", "PDB-derived RNA quality/structure metadata", "Quality weighting and filtering", "Metadata layer", "Filtering / benchmarking"],
]
df_registry = pd.DataFrame(registry_rows, columns=["dataset_name", "category", "purpose", "system_role", "math_layer", "usage_stage"])
registry_path = f"{DIRS['registry']}/DATASET_REGISTRY.csv"
df_registry.to_csv(registry_path, index=False)
display(df_registry)
print("Saved:", registry_path)

In [ ]:
# 10 — OPTIONAL NOTEBOOK REACTIVATION AUDIT

NB_PATH = HELIX_PATHS.get("notebook_current")
audit_rows = []

if NB_PATH and os.path.exists(NB_PATH):
    with open(NB_PATH, "r", encoding="utf-8", errors="ignore") as f:
        nb = json.load(f)
    for idx, cell in enumerate(nb.get("cells", []), start=1):
        if cell.get("cell_type") != "code":
            continue
        src = "".join(cell.get("source", []))
        imports, defs, assigns = [], [], []
        for line in src.splitlines():
            s = line.strip()
            if s.startswith("import ") or s.startswith("from "):
                imports.append(s)
            elif s.startswith("def ") or s.startswith("class "):
                defs.append(s)
            elif "=" in s and not s.startswith("#"):
                assigns.append(s[:160])
        audit_rows.append({
            "cell_index": idx,
            "char_len": len(src),
            "imports": imports[:12],
            "defs": defs[:12],
            "assigns": assigns[:12],
        })
df_nb_audit = pd.DataFrame(audit_rows)
audit_path = f"{DIRS['registry']}/NOTEBOOK_REACTIVATION_AUDIT.csv"
df_nb_audit.to_csv(audit_path, index=False)
display(df_nb_audit.head(10))
print("Saved:", audit_path)

In [ ]:
# 11 — FASTA DISCOVERY (DISABLED FOR LOW RAM)

print("⚠️ FASTA scanning disabled to prevent RAM crash")
df_fasta_manifest = None

In [ ]:
# 12 — METADATA NORMALIZATION + MERGE (LOW-RAM + ROBUST)

import gc

# 🔍 RESOLVE METADATA PATH
meta_path = HELIX_PATHS.get("metadata_csv")

if meta_path is None or not os.path.exists(meta_path):
    print("⚠️ Resolver failed — scanning fallback...")

    fallback_candidates = []
    for root in SEARCH_ROOTS:
        if not os.path.exists(root):
            continue

        for dirpath, _, filenames in os.walk(root):
            for f in filenames:
                if f.lower() == "rna_metadata.csv":
                    fallback_candidates.append(os.path.join(dirpath, f))

    if fallback_candidates:
        meta_path = sorted(fallback_candidates, key=lambda x: len(x))[0]
        print("✅ Found metadata:", meta_path)
    else:
        raise FileNotFoundError("❌ rna_metadata.csv not found anywhere")

# 📦 LOAD METADATA (LIMITED)
df_meta = pd.read_csv(meta_path)

if len(df_meta) > 200000:
    df_meta = df_meta.head(200000)

print("✅ Metadata loaded:", df_meta.shape)

# 🧬 NORMALIZE
df_meta = df_meta.copy()
df_meta["target_id_norm"] = df_meta["target_id"].astype(str)

if "pdb_id" in df_meta.columns:
    df_meta["pdb_id_norm"] = df_meta["pdb_id"].astype(str)
else:
    df_meta["pdb_id_norm"] = df_meta["target_id_norm"].str[:4]

# 🔥 REDUCE COLUMNS (CRITICAL FOR RAM)
keep_cols = [
    "pdb_id_norm",
    "resolution",
    "fraction_observed",
    "total_structuredness_adjusted",
    "missing_residues"
]

keep_cols = [c for c in keep_cols if c in df_meta.columns]
df_meta_small = df_meta[keep_cols].copy()

del df_meta
gc.collect()

# 🔗 PREP SOURCE
SOURCE = SOURCE.copy()
SOURCE["pdb_id_norm"] = SOURCE["target_id"].astype(str).str[:4]

# 🔗 MERGE
SOURCE_META = SOURCE.merge(
    df_meta_small,
    on="pdb_id_norm",
    how="left"
)

del df_meta_small
gc.collect()

print("✅ Metadata merged")
print("Shape:", SOURCE_META.shape)

In [ ]:
# 13 — QUALITY SCORE + FILTER

def build_quality_score(df):
    score = np.zeros(len(df), dtype=float)
    if "resolution" in df.columns:
        score += np.clip(5 - df["resolution"].fillna(5).astype(float), 0, 5)
    if "fraction_observed" in df.columns:
        score += df["fraction_observed"].fillna(0).astype(float)
    if "total_structuredness_adjusted" in df.columns:
        score += df["total_structuredness_adjusted"].fillna(0).astype(float)
    if "missing_residues" in df.columns:
        score -= df["missing_residues"].fillna(False).astype(int) * 0.5
    return score

SOURCE_META["quality_score"] = build_quality_score(SOURCE_META)

mask = pd.Series(True, index=SOURCE_META.index)
if "resolution" in SOURCE_META.columns:
    mask &= SOURCE_META["resolution"].fillna(10) < 4.5
if "fraction_observed" in SOURCE_META.columns:
    mask &= SOURCE_META["fraction_observed"].fillna(0) > 0.60

SOURCE_META_FILT = SOURCE_META[mask].copy()

print("✅ quality_score added + filter applied")
print("Before:", len(SOURCE_META), "After:", len(SOURCE_META_FILT))
print(SOURCE_META_FILT["quality_score"].describe())

In [ ]:
# 14 — STRUCTURE REGIME LABELS

def classify_structure_regime(df):
    if "total_structuredness_adjusted" not in df.columns:
        return np.array(["UNKNOWN"] * len(df), dtype=object)
    vals = df["total_structuredness_adjusted"].fillna(0).astype(float)
    conds = [vals > 0.7, vals > 0.4]
    choices = ["HIGH_STRUCTURE", "MEDIUM_STRUCTURE"]
    return np.select(conds, choices, default="LOW_STRUCTURE")

SOURCE_META_FILT["structure_regime"] = classify_structure_regime(SOURCE_META_FILT)
print("✅ structure_regime added")
print(SOURCE_META_FILT["structure_regime"].value_counts(dropna=False))

In [ ]:
# 15 — CORE GEOMETRY ENGINE

GEO = SOURCE_META_FILT.copy()

GEO["bond_length"] = np.sqrt(
    GEO["delta_dx"]**2 + GEO["delta_dy"]**2 + GEO["delta_dz"]**2
)

for col in ["prev_delta_dx", "prev_delta_dy", "prev_delta_dz"]:
    if col not in GEO.columns:
        base_col = "delta_dx" if col.endswith("_dx") else "delta_dy" if col.endswith("_dy") else "delta_dz"
        GEO[col] = GEO.groupby(["target_id", "chain", "copy"])[base_col].shift(1)

curr_vec = GEO[["delta_dx", "delta_dy", "delta_dz"]].values
prev_vec = GEO[["prev_delta_dx", "prev_delta_dy", "prev_delta_dz"]].fillna(0.0).values

dot = np.sum(curr_vec * prev_vec, axis=1)
curr_norm = np.linalg.norm(curr_vec, axis=1)
prev_norm = np.linalg.norm(prev_vec, axis=1)

GEO["cos_turn"] = np.clip(dot / (curr_norm * prev_norm + 1e-8), -1.0, 1.0)
GEO["turn_angle_rad"] = np.arccos(GEO["cos_turn"])
GEO["curvature_proxy"] = 1 - GEO["cos_turn"]

geo_core_path = f"{DIRS['geometry']}/GEOMETRY_CORE.parquet"
GEO.to_parquet(geo_core_path, index=False)
print("✅ Geometry core saved:", geo_core_path)
print(GEO.shape)

In [ ]:
# 16 — SECOND-ORDER GEOMETRY + TORSION PROXY

G2 = GEO.sort_values(["target_id", "chain", "copy", "residue_index"]).copy()

for ax in ["x_1", "y_1", "z_1"]:
    G2[f"{ax}_lead2"] = G2.groupby(["target_id", "chain", "copy"])[ax].shift(-2)

G2["i_to_i2_dist"] = np.sqrt(
    (G2["x_1_lead2"] - G2["x_1"])**2 +
    (G2["y_1_lead2"] - G2["y_1"])**2 +
    (G2["z_1_lead2"] - G2["z_1"])**2
)

for col in ["prev2_delta_dx", "prev2_delta_dy", "prev2_delta_dz"]:
    if col not in G2.columns:
        base_col = "delta_dx" if col.endswith("_dx") else "delta_dy" if col.endswith("_dy") else "delta_dz"
        G2[col] = G2.groupby(["target_id", "chain", "copy"])[base_col].shift(2)

v1 = G2[["prev2_delta_dx", "prev2_delta_dy", "prev2_delta_dz"]].fillna(0.0).values
v2 = G2[["prev_delta_dx", "prev_delta_dy", "prev_delta_dz"]].fillna(0.0).values
v3 = G2[["delta_dx", "delta_dy", "delta_dz"]].fillna(0.0).values

n1 = np.cross(v1, v2)
n2 = np.cross(v2, v3)

n1n = np.linalg.norm(n1, axis=1)
n2n = np.linalg.norm(n2, axis=1)

G2["torsion_proxy_cos"] = np.clip(np.sum(n1 * n2, axis=1) / (n1n * n2n + 1e-8), -1.0, 1.0)
G2["torsion_proxy_rad"] = np.arccos(G2["torsion_proxy_cos"])

g2_path = f"{DIRS['geometry']}/GEOMETRY_SECOND_ORDER.parquet"
G2.to_parquet(g2_path, index=False)
print("✅ Second-order geometry saved:", g2_path)

In [ ]:
# 17 — TOPOLOGY / SEGMENTS

TOP = G2.sort_values(["target_id", "chain", "copy", "residue_index"]).copy()

if "residue_gap_from_prev" not in TOP.columns:
    TOP["residue_gap_from_prev"] = TOP.groupby(["target_id", "chain", "copy"])["residue_index"].diff().fillna(1)

TOP["new_segment_flag"] = (
    (TOP["residue_gap_from_prev"] != 1) |
    TOP.groupby(["target_id", "chain", "copy"]).cumcount().eq(0)
).astype(int)

TOP["segment_id_local"] = TOP.groupby(["target_id", "chain", "copy"])["new_segment_flag"].cumsum()

seg_summary = (
    TOP.groupby(["target_id", "chain", "copy", "segment_id_local"])
    .agg(
        start_residue=("residue_index", "min"),
        end_residue=("residue_index", "max"),
        rows=("residue_index", "size"),
        mean_bond=("bond_length", "mean"),
        mean_turn=("turn_angle_rad", "mean"),
    )
    .reset_index()
)

top_rows_path = f"{DIRS['geometry']}/TOPOLOGY_ROWS.parquet"
top_seg_path = f"{DIRS['geometry']}/TOPOLOGY_SEGMENTS.parquet"
TOP.to_parquet(top_rows_path, index=False)
seg_summary.to_parquet(top_seg_path, index=False)

print("✅ Topology artifacts saved")
print(top_rows_path)
print(top_seg_path)

In [ ]:
# 18 — GEOMETRY CONFIG

geometry_config = {
    "bond_length_mean": float(GEO["bond_length"].mean()),
    "bond_length_median": float(GEO["bond_length"].median()),
    "bond_length_p90": float(np.percentile(GEO["bond_length"].dropna(), 90)),
    "turn_angle_mean_rad": float(GEO["turn_angle_rad"].mean()),
    "turn_angle_median_rad": float(GEO["turn_angle_rad"].median()),
    "i_to_i2_mean": float(G2["i_to_i2_dist"].dropna().mean()),
    "i_to_i2_median": float(G2["i_to_i2_dist"].dropna().median()),
    "torsion_proxy_mean_rad": float(G2["torsion_proxy_rad"].dropna().mean()),
    "torsion_proxy_median_rad": float(G2["torsion_proxy_rad"].dropna().median()),
    "recommended_bond_target": float(GEO["bond_length"].median()),
    "recommended_i2_target": float(G2["i_to_i2_dist"].dropna().median()),
    "recommended_curvature_weight": 0.50,
    "recommended_torsion_weight": 0.15,
}

config_path = f"{DIRS['geometry']}/geometry_config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(geometry_config, f, indent=2)

print("✅ geometry_config saved:", config_path)
print(json.dumps(geometry_config, indent=2))

In [ ]:
# 19 — LONG TRAJECTORY TABLE

GROUP_KEYS = ["target_id", "chain", "copy"]
traj_lengths = (
    GEO.groupby(GROUP_KEYS)
    .size()
    .reset_index(name="length")
    .sort_values("length", ascending=False)
    .reset_index(drop=True)
)

traj_path = f"{DIRS['benchmarks']}/LONG_TRAJECTORIES.csv"
traj_lengths.to_csv(traj_path, index=False)
display(traj_lengths.head(20))
print("Saved:", traj_path)

In [ ]:
# 20 — RECONSTRUCTION HELPERS

def get_group_df(source_df, target_id, chain, copy_):
    return (
        source_df[
            (source_df["target_id"] == target_id) &
            (source_df["chain"].astype(str) == str(chain)) &
            (source_df["copy"].astype(str) == str(copy_))
        ]
        .sort_values("residue_index")
        .reset_index(drop=True)
        .copy()
    )

def point_error(pred, true):
    return np.linalg.norm(pred - true[:len(pred)], axis=1)

def recon_naive(group):
    xyz = group[["x_1","y_1","z_1"]].values.astype(float)
    delta = group[["delta_dx","delta_dy","delta_dz"]].values.astype(float)
    cur = xyz[0].copy()
    out = [cur.copy()]
    for i in range(1, len(group)):
        cur = cur + delta[i]
        out.append(cur.copy())
    return np.vstack(out), xyz

def recon_anchor(group, reset_interval=20):
    xyz = group[["x_1","y_1","z_1"]].values.astype(float)
    delta = group[["delta_dx","delta_dy","delta_dz"]].values.astype(float)
    cur = xyz[0].copy()
    out = [cur.copy()]
    for i in range(1, len(group)):
        if i % reset_interval == 0:
            cur = xyz[i].copy()
        else:
            cur = cur + delta[i]
        out.append(cur.copy())
    return np.vstack(out), xyz

def recon_damped(group, alpha=0.85):
    xyz = group[["x_1","y_1","z_1"]].values.astype(float)
    delta = group[["delta_dx","delta_dy","delta_dz"]].values.astype(float)
    cur = xyz[0].copy()
    out = [cur.copy()]
    for i in range(1, len(group)):
        cur = cur + alpha * delta[i]
        out.append(cur.copy())
    return np.vstack(out), xyz

def recon_bond(group, bond_target=None):
    xyz = group[["x_1","y_1","z_1"]].values.astype(float)
    delta = group[["delta_dx","delta_dy","delta_dz"]].values.astype(float)
    bond_target = geometry_config["recommended_bond_target"] if bond_target is None else bond_target
    cur = xyz[0].copy()
    out = [cur.copy()]
    for i in range(1, len(group)):
        step = delta[i].copy()
        n = np.linalg.norm(step)
        if n > 1e-8:
            step = step / n * bond_target
        cur = cur + step
        out.append(cur.copy())
    return np.vstack(out), xyz

def recon_curvature(group, bond_target=None, curvature_weight=None):
    xyz = group[["x_1","y_1","z_1"]].values.astype(float)
    delta = group[["delta_dx","delta_dy","delta_dz"]].values.astype(float)
    bond_target = geometry_config["recommended_bond_target"] if bond_target is None else bond_target
    curvature_weight = geometry_config["recommended_curvature_weight"] if curvature_weight is None else curvature_weight

    cur = xyz[0].copy()
    prev_step = np.array([1.0, 0.0, 0.0], dtype=float)
    out = [cur.copy()]
    for i in range(1, len(group)):
        step = delta[i].copy()
        pred_u = step / (np.linalg.norm(step) + 1e-8)
        prev_u = prev_step / (np.linalg.norm(prev_step) + 1e-8)
        blended = (1 - curvature_weight) * pred_u + curvature_weight * prev_u
        blended = blended / (np.linalg.norm(blended) + 1e-8)
        step = blended * bond_target
        cur = cur + step
        prev_step = step.copy()
        out.append(cur.copy())
    return np.vstack(out), xyz

def recon_torsion(group, bond_target=None, curvature_weight=None, torsion_weight=None):
    xyz = group[["x_1","y_1","z_1"]].values.astype(float)
    delta = group[["delta_dx","delta_dy","delta_dz"]].values.astype(float)
    bond_target = geometry_config["recommended_bond_target"] if bond_target is None else bond_target
    curvature_weight = geometry_config["recommended_curvature_weight"] if curvature_weight is None else curvature_weight
    torsion_weight = geometry_config["recommended_torsion_weight"] if torsion_weight is None else torsion_weight

    cur = xyz[0].copy()
    prev_step = np.array([1.0, 0.0, 0.0], dtype=float)
    prev2_step = np.array([0.0, 1.0, 0.0], dtype=float)
    out = [cur.copy()]

    for i in range(1, len(group)):
        step = delta[i].copy()
        pred_u = step / (np.linalg.norm(step) + 1e-8)
        prev_u = prev_step / (np.linalg.norm(prev_step) + 1e-8)
        prev2_u = prev2_step / (np.linalg.norm(prev2_step) + 1e-8)

        blended = (1 - curvature_weight) * pred_u + curvature_weight * prev_u

        n1 = np.cross(prev2_u, prev_u)
        n2 = np.cross(prev_u, pred_u)
        if np.linalg.norm(n1) > 1e-8 and np.linalg.norm(n2) > 1e-8:
            normal_blend = (n1 / (np.linalg.norm(n1) + 1e-8) + n2 / (np.linalg.norm(n2) + 1e-8)) / 2.0
            blended = blended + torsion_weight * normal_blend

        blended = blended / (np.linalg.norm(blended) + 1e-8)
        step = blended * bond_target

        cur = cur + step
        out.append(cur.copy())
        prev2_step = prev_step.copy()
        prev_step = step.copy()

    return np.vstack(out), xyz

In [ ]:
# 21 — SINGLE LONG-TRAJECTORY BENCHMARK

chosen = traj_lengths.iloc[0]
g_long = get_group_df(GEO, chosen["target_id"], chosen["chain"], chosen["copy"])

modes = {
    "naive": recon_naive,
    "anchored": recon_anchor,
    "damped": recon_damped,
    "bond": recon_bond,
    "curvature": recon_curvature,
    "torsion": recon_torsion,
}

bench_rows = []
results = {}
for mode, fn in modes.items():
    pred, true = fn(g_long)
    e = point_error(pred, true)
    results[mode] = {"pred": pred, "true": true, "errors": e}
    bench_rows.append({
        "mode": mode,
        "mean": float(e.mean()),
        "median": float(np.median(e)),
        "p90": float(np.percentile(e, 90)),
        "max": float(e.max()),
        "length": int(len(e)),
    })

df_bench = pd.DataFrame(bench_rows).sort_values("mean").reset_index(drop=True)
bench_path = f"{DIRS['benchmarks']}/RECON_BENCHMARKS_SINGLE.csv"
df_bench.to_csv(bench_path, index=False)
display(df_bench)
print("Saved:", bench_path)

In [ ]:
# 22 — TOP-N LONG-HORIZON STRESS TEST

topN = traj_lengths.head(10)
stress_rows = []

for _, r in topN.iterrows():
    gdf = get_group_df(GEO, r["target_id"], r["chain"], r["copy"])
    row = {
        "target_id": r["target_id"],
        "chain": r["chain"],
        "copy": r["copy"],
        "length": int(r["length"]),
    }
    for mode in ["naive", "bond", "curvature", "torsion"]:
        pred, true = modes[mode](gdf)
        e = point_error(pred, true)
        row[f"{mode}_mean"] = float(e.mean())
        row[f"{mode}_p90"] = float(np.percentile(e, 90))
        row[f"{mode}_max"] = float(e.max())
        row[f"{mode}_final"] = float(e[-1])
    stress_rows.append(row)

df_stress = pd.DataFrame(stress_rows)
stress_path = f"{DIRS['benchmarks']}/LONG_HORIZON_STRESS.csv"
df_stress.to_csv(stress_path, index=False)
display(df_stress)
print("Saved:", stress_path)

In [ ]:
# 23 — SEGMENT-AWARE RECONSTRUCTION

def recon_segment_aware(group, mode="torsion"):
    xyz = group[["x_1", "y_1", "z_1"]].values.astype(float)
    delta = group[["delta_dx", "delta_dy", "delta_dz"]].values.astype(float)
    gaps = group["residue_gap_from_prev"].fillna(1).values if "residue_gap_from_prev" in group.columns else np.ones(len(group))

    cur = xyz[0].copy()
    prev_step = np.array([1.0, 0.0, 0.0], dtype=float)
    prev2_step = np.array([0.0, 1.0, 0.0], dtype=float)
    out = [cur.copy()]

    for i in range(1, len(group)):
        if gaps[i] != 1:
            cur = xyz[i].copy()
            out.append(cur.copy())
            prev_step = np.array([1.0, 0.0, 0.0], dtype=float)
            prev2_step = np.array([0.0, 1.0, 0.0], dtype=float)
            continue

        step = delta[i].copy()
        bond_target = geometry_config["recommended_bond_target"]

        if mode == "bond":
            step = step / (np.linalg.norm(step) + 1e-8) * bond_target
        elif mode in ["curvature", "torsion"]:
            pred_u = step / (np.linalg.norm(step) + 1e-8)
            prev_u = prev_step / (np.linalg.norm(prev_step) + 1e-8)
            blended = (1 - geometry_config["recommended_curvature_weight"]) * pred_u + geometry_config["recommended_curvature_weight"] * prev_u
            if mode == "torsion":
                prev2_u = prev2_step / (np.linalg.norm(prev2_step) + 1e-8)
                n1 = np.cross(prev2_u, prev_u)
                n2 = np.cross(prev_u, pred_u)
                if np.linalg.norm(n1) > 1e-8 and np.linalg.norm(n2) > 1e-8:
                    blended = blended + geometry_config["recommended_torsion_weight"] * (
                        (n1/(np.linalg.norm(n1)+1e-8) + n2/(np.linalg.norm(n2)+1e-8)) / 2
                    )
            step = blended / (np.linalg.norm(blended) + 1e-8) * bond_target

        cur = cur + step
        out.append(cur.copy())
        prev2_step = prev_step.copy()
        prev_step = step.copy()

    pred = np.vstack(out)
    e = point_error(pred, xyz)
    return pred, xyz, e

seg_rows = []
for _, r in topN.iterrows():
    gdf = get_group_df(GEO, r["target_id"], r["chain"], r["copy"])
    for mode in ["bond", "curvature", "torsion"]:
        pred, true, e = recon_segment_aware(gdf, mode=mode)
        seg_rows.append({
            "target_id": r["target_id"],
            "chain": r["chain"],
            "copy": r["copy"],
            "length": int(r["length"]),
            "mode": mode,
            "mean": float(e.mean()),
            "median": float(np.median(e)),
            "p90": float(np.percentile(e, 90)),
            "max": float(e.max()),
        })

df_seg = pd.DataFrame(seg_rows)
seg_path = f"{DIRS['benchmarks']}/SEGMENT_AWARE_BENCHMARK.csv"
df_seg.to_csv(seg_path, index=False)
display(df_seg.head(20))
print("Saved:", seg_path)

In [ ]:
# 24 — DIRECT COORDINATE MODEL DIAGNOSTIC (SAFE VERSION)

print("📊 Checking direct coordinate diagnostic...")

if "df_pred_coord" in globals() and df_pred_coord is not None:
    if "euclid_abs_err" in df_pred_coord.columns:
        e = df_pred_coord["euclid_abs_err"].values

        df_coord_diag = pd.DataFrame([{
            "mean": float(e.mean()),
            "median": float(np.median(e)),
            "p90": float(np.percentile(e, 90)),
        }])

        display(df_coord_diag)
        print("✅ Coordinate diagnostic complete")

    else:
        print("⚠️ df_pred_coord exists but missing 'euclid_abs_err'")
else:
    print("⚠️ Skipping — df_pred_coord not loaded (not required)")

In [ ]:
# 25 — FEATURE MATRIX + GLOBAL FEATURE CONTRACT (AUTHORITATIVE)

import numpy as np
import pandas as pd
import gc

print("🧬 Building feature system (GLOBAL CONTRACT)")

assert "SOURCE" in globals(), "❌ SOURCE not defined"

df = SOURCE.copy()

# 🔹 REQUIRED TARGETS
for c in ["delta_dx", "delta_dy", "delta_dz"]:
    assert c in df.columns, f"❌ Missing {c}"

# 🔹 POSITION
if "residue_index" not in df.columns:
    df["residue_index"] = np.arange(len(df))

df["res_idx_norm"] = df["residue_index"] / (df["residue_index"].max() + 1)

# 🔹 LOCAL CONTEXT
for axis in ["delta_dx", "delta_dy", "delta_dz"]:
    df[f"{axis}_prev"] = df.groupby("target_id")[axis].shift(1).fillna(0.0)
    df[f"{axis}_next"] = df.groupby("target_id")[axis].shift(-1).fillna(0.0)

# 🔹 OPTIONAL GEOMETRY (SAFE DEFAULTS)
for col in ["curvature", "torsion", "bond_length"]:
    if col not in df.columns:
        df[col] = 0.0

# 🔥 GLOBAL FEATURE LIST (THIS FIXES YOUR ERROR)
feature_cols = [
    "res_idx_norm",
    "curvature",
    "torsion",
    "bond_length",

    "delta_dx_prev",
    "delta_dy_prev",
    "delta_dz_prev",

    "delta_dx_next",
    "delta_dy_next",
    "delta_dz_next",
]

# 🔹 BUILD MATRICES
X = df[feature_cols].fillna(0.0).values.astype(np.float32)
Y = df[["delta_dx", "delta_dy", "delta_dz"]].values.astype(np.float32)
W = np.ones(len(df), dtype=np.float32)

print("✅ feature_cols:", len(feature_cols))
print("✅ X:", X.shape)
print("✅ Y:", Y.shape)

# 🔹 CLEANUP
del df
gc.collect()

In [ ]:
# 26 — RANDOM FOREST DELTA MODEL (LOW-RAM SAFE VERSION)

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import gc

# 🔻 HARD LIMIT DATA SIZE (CRITICAL)
MAX_ROWS = 100000

if len(X) > MAX_ROWS:
    print(f"⚠️ Reducing dataset from {len(X)} → {MAX_ROWS}")
    idx = np.random.choice(len(X), MAX_ROWS, replace=False)
    X_small = X[idx]
    Y_small = Y[idx]
    W_small = W[idx]
else:
    X_small, Y_small, W_small = X, Y, W

# 🔻 TRAIN/VALID SPLIT
X_train, X_valid, Y_train, Y_valid, W_train, W_valid = train_test_split(
    X_small, Y_small, W_small,
    test_size=0.2,
    random_state=42
)

# 🔻 MEMORY-SAFE MODELS
rf_dx = RandomForestRegressor(
    n_estimators=40,      # reduced
    max_depth=12,         # limit tree size
    n_jobs=1,             # 🚨 CRITICAL FIX (no parallel explosion)
    random_state=42
)

rf_dy = RandomForestRegressor(
    n_estimators=40,
    max_depth=12,
    n_jobs=1,
    random_state=42
)

rf_dz = RandomForestRegressor(
    n_estimators=40,
    max_depth=12,
    n_jobs=1,
    random_state=42
)

print("🚀 Training Random Forest (safe mode)...")

rf_dx.fit(X_train, Y_train[:, 0], sample_weight=W_train)
rf_dy.fit(X_train, Y_train[:, 1], sample_weight=W_train)
rf_dz.fit(X_train, Y_train[:, 2], sample_weight=W_train)

# 🔻 FREE TRAIN MEMORY EARLY
del X_train, Y_train, W_train
gc.collect()

# 🔻 VALIDATION
pred_valid = np.column_stack([
    rf_dx.predict(X_valid),
    rf_dy.predict(X_valid),
    rf_dz.predict(X_valid),
])

delta_err = np.linalg.norm(pred_valid - Y_valid, axis=1)

df_rf_diag = pd.DataFrame([{
    "mean": float(delta_err.mean()),
    "median": float(np.median(delta_err)),
    "p90": float(np.percentile(delta_err, 90)),
    "rows_valid": int(len(delta_err)),
}])

rf_diag_path = f"{DIRS['models']}/RF_DELTA_DIAGNOSTIC_SAFE.csv"
df_rf_diag.to_csv(rf_diag_path, index=False)

display(df_rf_diag)
print("✅ Saved:", rf_diag_path)

# 🔻 CLEANUP
del X_valid, Y_valid, W_valid, pred_valid, delta_err
gc.collect()

In [ ]:
# 27 — HISTGRADIENTBOOSTING (SELF-CONTAINED + LOW RAM SAFE)

from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
import numpy as np
import gc

print("🧬 Running HGB (self-contained mode)...")

# 🔻 SAFETY CHECK
assert "X" in globals(), "❌ X not defined — run feature cell first"
assert "Y" in globals(), "❌ Y not defined — run feature cell first"
assert "W" in globals(), "❌ W not defined — run feature cell first"

# 🔻 LIMIT SIZE (CRITICAL FOR RAM)
MAX_ROWS = 100000

if len(X) > MAX_ROWS:
    print(f"⚠️ Reducing dataset from {len(X)} → {MAX_ROWS}")
    idx = np.random.choice(len(X), MAX_ROWS, replace=False)
    X_small = X[idx]
    Y_small = Y[idx]
    W_small = W[idx]
else:
    X_small, Y_small, W_small = X, Y, W

# 🔻 TRAIN/VALID SPLIT (DEFINED HERE — FIXES YOUR ERROR)
X_train, X_valid, Y_train, Y_valid, W_train, W_valid = train_test_split(
    X_small, Y_small, W_small,
    test_size=0.2,
    random_state=42
)

# 🔻 MODEL (MEMORY SAFE)
hgb_dx = HistGradientBoostingRegressor(max_iter=120, random_state=42)
hgb_dy = HistGradientBoostingRegressor(max_iter=120, random_state=42)
hgb_dz = HistGradientBoostingRegressor(max_iter=120, random_state=42)

print("🚀 Training HGB...")

hgb_dx.fit(X_train, Y_train[:, 0])
hgb_dy.fit(X_train, Y_train[:, 1])
hgb_dz.fit(X_train, Y_train[:, 2])

# 🔻 VALIDATION
pred_valid = np.column_stack([
    hgb_dx.predict(X_valid),
    hgb_dy.predict(X_valid),
    hgb_dz.predict(X_valid),
])

delta_err = np.linalg.norm(pred_valid - Y_valid, axis=1)

df_hgb_diag = pd.DataFrame([{
    "mean": float(delta_err.mean()),
    "median": float(np.median(delta_err)),
    "p90": float(np.percentile(delta_err, 90)),
    "rows": int(len(delta_err)),
}])

display(df_hgb_diag)

# 🔻 CLEANUP (VERY IMPORTANT)
del X_train, X_valid, Y_train, Y_valid, W_train, W_valid
del pred_valid, delta_err
gc.collect()

print("✅ HGB complete (memory safe)")

In [ ]:
# 28 — RECONSTRUCT FROM MODEL PREDICTIONS

def prepare_group_features(group, feature_cols):
    return group[feature_cols].fillna(0).values

def reconstruct_from_model(group, model_triplet, mode="torsion"):
    model_x, model_y, model_z = model_triplet
    xyz = group[["x_1", "y_1", "z_1"]].values.astype(float)
    Xg = prepare_group_features(group, feature_cols)
    pred_delta = np.column_stack([
        model_x.predict(Xg),
        model_y.predict(Xg),
        model_z.predict(Xg),
    ])

    cur = xyz[0].copy()
    prev_step = np.array([1.0, 0.0, 0.0], dtype=float)
    prev2_step = np.array([0.0, 1.0, 0.0], dtype=float)
    out = [cur.copy()]

    gaps = group["residue_gap_from_prev"].fillna(1).values if "residue_gap_from_prev" in group.columns else np.ones(len(group))

    for i in range(1, len(group)):
        if gaps[i] != 1:
            cur = xyz[i].copy()
            out.append(cur.copy())
            prev_step = np.array([1.0, 0.0, 0.0], dtype=float)
            prev2_step = np.array([0.0, 1.0, 0.0], dtype=float)
            continue

        step = pred_delta[i].copy()
        bond_target = geometry_config["recommended_bond_target"]

        pred_u = step / (np.linalg.norm(step) + 1e-8)
        prev_u = prev_step / (np.linalg.norm(prev_step) + 1e-8)
        blended = (1 - geometry_config["recommended_curvature_weight"]) * pred_u + geometry_config["recommended_curvature_weight"] * prev_u

        if mode == "torsion":
            prev2_u = prev2_step / (np.linalg.norm(prev2_step) + 1e-8)
            n1 = np.cross(prev2_u, prev_u)
            n2 = np.cross(prev_u, pred_u)
            if np.linalg.norm(n1) > 1e-8 and np.linalg.norm(n2) > 1e-8:
                blended = blended + geometry_config["recommended_torsion_weight"] * (
                    (n1 / (np.linalg.norm(n1) + 1e-8) + n2 / (np.linalg.norm(n2) + 1e-8)) / 2
                )

        step = blended / (np.linalg.norm(blended) + 1e-8) * bond_target
        cur = cur + step
        out.append(cur.copy())
        prev2_step = prev_step.copy()
        prev_step = step.copy()

    pred = np.vstack(out)
    err = point_error(pred, xyz)
    return pred, xyz, err

In [ ]:
# 20 — RECONSTRUCTION HELPERS (FULL FIXED VERSION — FEATURE SAFE)

import numpy as np

# 🔹 FEATURE BUILDER (CRITICAL FIX)
def prepare_group_features(group, feature_cols):
    g = group.copy()

    # 🔹 ENSURE ORDER
    g = g.sort_values("residue_index").reset_index(drop=True)

    # 🔹 POSITION FEATURE
    if "residue_index" not in g.columns:
        g["residue_index"] = np.arange(len(g))

    g["res_idx_norm"] = g["residue_index"] / (g["residue_index"].max() + 1)

    # 🔹 LOCAL DELTA CONTEXT
    for axis in ["delta_dx", "delta_dy", "delta_dz"]:
        if axis not in g.columns:
            g[axis] = 0.0

        g[f"{axis}_prev"] = g[axis].shift(1).fillna(0.0)
        g[f"{axis}_next"] = g[axis].shift(-1).fillna(0.0)

    # 🔹 GEOMETRY DEFAULTS (SAFE)
    for col in ["curvature", "torsion", "bond_length"]:
        if col not in g.columns:
            g[col] = 0.0

    # 🔹 FINAL GUARANTEE (NO KEYERROR EVER AGAIN)
    for col in feature_cols:
        if col not in g.columns:
            g[col] = 0.0

    return g[feature_cols].fillna(0.0).values.astype(np.float32)


# 🔹 RECONSTRUCTION FROM MODEL
def reconstruct_from_model(group, model_triplet, mode="torsion"):
    model_x, model_y, model_z = model_triplet

    # 🔹 TRUE COORDS
    xyz = group[["x_1", "y_1", "z_1"]].values.astype(float)

    # 🔹 BUILD FEATURES (FIXED)
    Xg = prepare_group_features(group, feature_cols)

    # 🔹 PREDICT DELTAS
    pred_delta = np.column_stack([
        model_x.predict(Xg),
        model_y.predict(Xg),
        model_z.predict(Xg),
    ])

    # 🔹 INIT
    cur = xyz[0].copy()
    prev_step = np.array([1.0, 0.0, 0.0], dtype=float)
    prev2_step = np.array([0.0, 1.0, 0.0], dtype=float)

    coords = [cur.copy()]

    for i in range(1, len(group)):
        step = pred_delta[i].copy()

        # 🔹 NORMALIZE STEP
        pred_u = step / (np.linalg.norm(step) + 1e-8)
        prev_u = prev_step / (np.linalg.norm(prev_step) + 1e-8)

        # 🔹 CURVATURE BLEND
        blended = (
            (1 - geometry_config["recommended_curvature_weight"]) * pred_u +
            geometry_config["recommended_curvature_weight"] * prev_u
        )

        # 🔹 TORSION ADJUSTMENT
        prev2_u = prev2_step / (np.linalg.norm(prev2_step) + 1e-8)
        n1 = np.cross(prev2_u, prev_u)
        n2 = np.cross(prev_u, pred_u)

        if np.linalg.norm(n1) > 1e-8 and np.linalg.norm(n2) > 1e-8:
            blended += geometry_config["recommended_torsion_weight"] * (
                (n1 / (np.linalg.norm(n1) + 1e-8) +
                 n2 / (np.linalg.norm(n2) + 1e-8)) / 2
            )

        # 🔹 NORMALIZE + SCALE
        blended = blended / (np.linalg.norm(blended) + 1e-8)
        step = blended * geometry_config["recommended_bond_target"]

        # 🔹 APPLY STEP
        cur = cur + step
        coords.append(cur.copy())

        # 🔹 UPDATE HISTORY
        prev2_step = prev_step.copy()
        prev_step = step.copy()

    pred = np.vstack(coords)

    # 🔹 ERROR
    err = np.linalg.norm(pred - xyz[:len(pred)], axis=1)

    return pred, xyz, err

In [ ]:
# 30 — COMPETITION SEQUENCE → FEATURE ROWS

def sequence_to_feature_rows(seq, target_id):
    rows = []
    base_map = {"A": [1,0,0,0], "C": [0,1,0,0], "G": [0,0,1,0], "U": [0,0,0,1]}
    n = len(seq)

    for i, b in enumerate(seq):
        base_A, base_C, base_G, base_U = base_map.get(b, [0,0,0,0])
        rows.append({
            "target_id": target_id,
            "residue_index_0based": i,
            "residue_frac": i / max(n - 1, 1),
            "segment_frac": i / max(n - 1, 1),
            "segment_residue_rank0": i,
            "is_segment_start": 1 if i == 0 else 0,
            "is_segment_end": 1 if i == n - 1 else 0,
            "base_A": base_A, "base_C": base_C, "base_G": base_G, "base_U": base_U,
            "prev_delta_dx": 0.0, "prev_delta_dy": 0.0, "prev_delta_dz": 0.0,
            "prev2_delta_dx": 0.0, "prev2_delta_dy": 0.0, "prev2_delta_dz": 0.0,
            "prev_delta_norm": 0.0, "prev2_delta_norm": 0.0,
            "prev_cos_prev2": 0.0, "prev_delta_change_norm": 0.0,
            "quality_score": 1.0,
        })
    return pd.DataFrame(rows)

In [ ]:
# 31 — PREDICT COORDS FOR A SINGLE TEST SEQUENCE (FULL FIXED VERSION)

import numpy as np
import pandas as pd

def predict_sequence_coords(seq, target_id, model_triplet):
    model_x, model_y, model_z = model_triplet

    n = len(seq)

    # 🔹 BUILD BASE FRAME
    g = pd.DataFrame({
        "target_id": [target_id] * n,
        "residue_index": np.arange(n)
    })

    # 🔹 POSITION FEATURE
    g["res_idx_norm"] = g["residue_index"] / max(n - 1, 1)

    # 🔹 REQUIRED FEATURE SCAFFOLD (CRITICAL FIX)
    for col in feature_cols:
        if col not in g.columns:
            g[col] = 0.0

    # 🔹 LOCAL CONTEXT (initialize as zeros)
    for axis in ["delta_dx", "delta_dy", "delta_dz"]:
        g[f"{axis}_prev"] = 0.0
        g[f"{axis}_next"] = 0.0

    # 🔹 GEOMETRY PLACEHOLDERS
    for col in ["curvature", "torsion", "bond_length"]:
        g[col] = 0.0

    # 🔹 FINAL FEATURE MATRIX
    Xg = g[feature_cols].fillna(0.0).values.astype(np.float32)

    # 🔹 PREDICT DELTAS
    pred_delta = np.column_stack([
        model_x.predict(Xg),
        model_y.predict(Xg),
        model_z.predict(Xg),
    ])

    # 🔹 RECONSTRUCTION
    cur = np.array([0.0, 0.0, 0.0], dtype=float)
    prev_step = np.array([1.0, 0.0, 0.0], dtype=float)
    prev2_step = np.array([0.0, 1.0, 0.0], dtype=float)

    coords = [cur.copy()]

    for i in range(1, n):
        step = pred_delta[i].copy()

        pred_u = step / (np.linalg.norm(step) + 1e-8)
        prev_u = prev_step / (np.linalg.norm(prev_step) + 1e-8)

        blended = (
            (1 - geometry_config["recommended_curvature_weight"]) * pred_u +
            geometry_config["recommended_curvature_weight"] * prev_u
        )

        prev2_u = prev2_step / (np.linalg.norm(prev2_step) + 1e-8)
        n1 = np.cross(prev2_u, prev_u)
        n2 = np.cross(prev_u, pred_u)

        if np.linalg.norm(n1) > 1e-8 and np.linalg.norm(n2) > 1e-8:
            blended += geometry_config["recommended_torsion_weight"] * (
                (n1/(np.linalg.norm(n1)+1e-8) +
                 n2/(np.linalg.norm(n2)+1e-8)) / 2
            )

        blended = blended / (np.linalg.norm(blended) + 1e-8)
        step = blended * geometry_config["recommended_bond_target"]

        cur = cur + step
        coords.append(cur.copy())

        prev2_step = prev_step.copy()
        prev_step = step.copy()

    return np.vstack(coords)

In [ ]:
# 32 — BUILD 5-TRACK COMPETITION SUBMISSION

test_df = COMP_DATA.get("test_sequences", None)
sample_sub = COMP_DATA.get("sample_submission", None)

if test_df is None or sample_sub is None:
    print("⚠️ Missing test_sequences or sample_submission; skipping submission build")
else:
    submission_rows = []

    for _, row in test_df.iterrows():
        target_id = row["target_id"]
        seq = row["sequence"]

        coords1 = predict_sequence_coords(seq, target_id, rf_triplet)
        coords2 = predict_sequence_coords(seq, target_id, hgb_triplet)
        coords3 = coords2 + np.linspace(0, 0.2, len(coords2))[:, None] * np.array([1.0, 0.0, 0.0])
        coords4 = coords2 + np.linspace(0, 0.2, len(coords2))[:, None] * np.array([0.0, 1.0, 0.0])

        coords5 = np.zeros((len(seq), 3))
        for i in range(len(seq)):
            ang = i * 0.6
            coords5[i] = [10 * np.cos(ang), 10 * np.sin(ang), i * 2.5]

        for i, base in enumerate(seq):
            submission_rows.append({
                "ID": f"{target_id}_{i+1}",
                "resname": base,
                "resid": i+1,
                "x_1": float(coords1[i, 0]), "y_1": float(coords1[i, 1]), "z_1": float(coords1[i, 2]),
                "x_2": float(coords2[i, 0]), "y_2": float(coords2[i, 1]), "z_2": float(coords2[i, 2]),
                "x_3": float(coords3[i, 0]), "y_3": float(coords3[i, 1]), "z_3": float(coords3[i, 2]),
                "x_4": float(coords4[i, 0]), "y_4": float(coords4[i, 1]), "z_4": float(coords4[i, 2]),
                "x_5": float(coords5[i, 0]), "y_5": float(coords5[i, 1]), "z_5": float(coords5[i, 2]),
            })

    submission_df = pd.DataFrame(submission_rows)
    submission_df = submission_df[sample_sub.columns.tolist()]
    submission_path = f"{DIRS['submission']}/submission.csv"
    submission_df.to_csv(submission_path, index=False)
    print("✅ submission.csv written:", submission_path)
    print(submission_df.head())

In [ ]:
# 33 — EXPORTS + FINAL INDEX

import pickle

# model bundle
bundle = {
    "feature_cols": feature_cols,
    "rf_dx": rf_dx, "rf_dy": rf_dy, "rf_dz": rf_dz,
    "hgb_dx": hgb_dx, "hgb_dy": hgb_dy, "hgb_dz": hgb_dz,
    "geometry_config": geometry_config,
}
bundle_path = f"{DIRS['models']}/HELIX_MODELS_BUNDLE.pkl"
with open(bundle_path, "wb") as f:
    pickle.dump(bundle, f)

# loader
loader_code = f'''
import os, json, pickle, pandas as pd

PROJECT_ROOT = r"{PROJECT_ROOT}"

def load_geometry_config():
    with open(f"{{PROJECT_ROOT}}/08_GEOMETRY_ENGINE/geometry_config.json", "r") as f:
        return json.load(f)

def load_model_bundle():
    with open(f"{{PROJECT_ROOT}}/12_MODELS/HELIX_MODELS_BUNDLE.pkl", "rb") as f:
        return pickle.load(f)

def load_delta_table():
    return pd.read_parquet(f"{{PROJECT_ROOT}}/02_DELTA_TRUTH/BASELINE_DELTA_TABLE_CLEAN_STRICT.parquet")
'''
loader_path = f"{DIRS['exports']}/helix_canonical_loader.py"
with open(loader_path, "w", encoding="utf-8") as f:
    f.write(loader_code)

# summary
summary = {
    "project_root": PROJECT_ROOT,
    "competition_files_loaded": {k: None if v is None else list(v.shape) for k, v in COMP_DATA.items()},
    "geometry_config": geometry_config,
    "top_long_trajectory": None if len(traj_lengths) == 0 else traj_lengths.iloc[0].to_dict(),
}
summary_path = f"{DIRS['exports']}/HELIX_NOTEBOOK_SUMMARY.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, default=str)

# index
index_rows = []
for folder_key, folder_path in DIRS.items():
    file_count, total_mb = 0, 0.0
    for dirpath, dirnames, filenames in os.walk(folder_path):
        for fn in filenames:
            file_count += 1
            fp = os.path.join(dirpath, fn)
            try:
                total_mb += os.path.getsize(fp) / (1024 * 1024)
            except Exception:
                pass
    index_rows.append({
        "folder_key": folder_key,
        "folder_path": folder_path,
        "file_count": file_count,
        "total_mb": round(total_mb, 3),
    })

df_index = pd.DataFrame(index_rows)
index_path = f"{DIRS['exports']}/HELIX_PROJECT_INDEX.csv"
df_index.to_csv(index_path, index=False)

print("✅ Final exports complete")
print("Model bundle:", bundle_path)
print("Loader:", loader_path)
print("Summary:", summary_path)
print("Index:", index_path)
display(df_index)